# Cell 0 - vertex ai



In [ ]:
# ============================================================
# CELL 0: VERTEX AI SETUP & CONNECTION TEST
# ============================================================
!pip install -q google-cloud-aiplatform openai

import json, tempfile, os
from google.colab import userdata
import google.auth
import google.auth.transport.requests
from google.oauth2 import service_account
from openai import OpenAI

# Load service account credentials
service_account_info = json.loads(userdata.get("GCP_SERVICE_ACCOUNT_JSON"))

# Get access token from service account
credentials = service_account.Credentials.from_service_account_info(
    service_account_info,
    scopes=["https://www.googleapis.com/auth/cloud-platform"]
)
auth_request = google.auth.transport.requests.Request()
credentials.refresh(auth_request)
access_token = credentials.token

# Vertex AI config
PROJECT_ID = "nth-celerity-488821-b3"
LOCATION = "us-east5"
VERTEX_MODEL_ID = "meta/llama-4-maverick-17b-128e-instruct-maas"

vertex_client = OpenAI(
    base_url=f"https://{LOCATION}-aiplatform.googleapis.com/v1/projects/{PROJECT_ID}/locations/{LOCATION}/endpoints/openapi",
    api_key=access_token
)

# Test connection
print("⏳ Testing Vertex AI Llama 4 connection...")
response = vertex_client.chat.completions.create(
    model=VERTEX_MODEL_ID,
    messages=[{"role": "user", "content": "Reply with just: OK"}],
    max_tokens=5
)
print(f"   Response: {response.choices[0].message.content.strip()}")
print("✅ Vertex AI Llama 4 Maverick connected successfully")

def refresh_vertex_credentials():
    """
    Refresh the Vertex AI access token.
    Call this if you get a 401 UNAUTHENTICATED error.
    Token expires after 1 hour in Colab.
    On GCP this is handled automatically — this is Colab-only.
    """
    global vertex_client, access_token

    credentials.refresh(auth_request)
    access_token = credentials.token

    vertex_client = OpenAI(
        base_url=f"https://{LOCATION}-aiplatform.googleapis.com/v1"
                 f"/projects/{PROJECT_ID}/locations/{LOCATION}"
                 f"/endpoints/openapi",
        api_key=access_token
    )
    print("✅ Vertex AI credentials refreshed")

print("💡 If you get a 401 error, run: refresh_vertex_credentials()")

⏳ Testing Vertex AI Llama 4 connection...
   Response: OK
✅ Vertex AI Llama 4 Maverick connected successfully
💡 If you get a 401 error, run: refresh_vertex_credentials()


# Cell 1 - dependencies - run once per session

In [ ]:
# ============================================================
# CELL 1: INSTALL ALL DEPENDENCIES
# ============================================================
# %%capture

# Step 1: Fix packaging version FIRST — PaddleOCR requires this
!pip install "packaging==23.2" -q

# Step 2: Clean slate for conflicting packages
!pip uninstall -y numpy paddlepaddle paddleocr torch transformers sentencepiece 2>/dev/null

# Step 3: Install numpy first — everything depends on it
!pip install numpy==1.26.4 -q

# Step 4: Core PDF and image libraries
!pip install pymupdf Pillow scipy -q

# Step 5: PaddlePaddle and PaddleOCR — order matters
!pip install paddlepaddle==2.6.2 -q
!pip install paddleocr==2.8.1 -q

# Step 6: PyTorch and transformers
!pip install torch==2.2.2 -q
!pip install transformers==4.38.2 sentencepiece -q

# Step 7: Remaining dependencies
!pip install groq -q
!pip install presidio-analyzer presidio-anonymizer -q
!pip install sacrebleu -q
!pip install bert-score -q
!pip install google-cloud-aiplatform openai -q

# Step 8: spaCy model
!python -m spacy download en_core_web_lg -q

print("\n" + "="*60)
print("✅ All dependencies installed")
print("="*60)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 53.0/53.0 kB 2.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
xarray 2025.12.0 requires packaging>=24.1, but you have packaging 23.2 which is incompatible.
db-dtypes 1.5.0 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
google-cloud-bigquery 3.40.1 requires packaging>=24.2.0, but you have packaging 23.2 which is incompatible.
wheel 0.46.3 requires packaging>=24.0, but you have packaging 23.2 which is incompatible.
Found existing installation: numpy 2.0.2
Uninstalling numpy-2.0.2:
  Successfully uninstalled numpy-2.0.2
Found existing installation: torch 2.10.0+cu128
Uninstalling torch-2.10.0+cu128:
  Successfully uninstalled torch-2.10.0+cu128
Found existing installation: transformers 5.0.0
Uninstalling transformers-5.0.0:
  Successfully uninstalled transformers-5.0.0


# Cell 2 - imports

In [ ]:
# ============================================================
# CELL 2: IMPORTS & GLOBAL CONFIGURATION
# ============================================================
import os, re, json, time, tempfile
import numpy as np
import torch
import pymupdf
from PIL import Image, ImageDraw
from scipy.ndimage import sobel
from IPython.display import display, Image as IPImage
# import pytesseract

os.environ["PADDLE_PDX_DISABLE_MODEL_SOURCE_CHECK"] = "True"

# PaddleOCR
PaddleOCR = None
try:
    from paddleocr import PaddleOCR
    print("✅ PaddleOCR: available")
except Exception as e:
    print(f"⚠️  PaddleOCR unavailable: {e} — will fall back to Tesseract")

# Global constants
OCR_DPI              = 300
OCR_CONFIDENCE       = 0.50
WORK_DIR             = "/content/courtaccess_output"
os.makedirs(WORK_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"\n✅ Device: {device}")
print("✅ All imports ready")



✅ PaddleOCR: available

✅ Device: cuda
✅ All imports ready


# Cell 3 - pdf upload

In [ ]:

# ============================================================
# CELL 3: UPLOAD PDF
# ============================================================
from pathlib import Path
from google.colab import files

print("📤 Upload a PDF to translate:")
try:
    uploaded = files.upload()
    if uploaded:
        fname = list(uploaded.keys())[0]
        pdf_path = Path(WORK_DIR) / fname
        pdf_path.write_bytes(uploaded[fname])
        INPUT_PDF = str(pdf_path)
        print(f"✅ Uploaded: {INPUT_PDF}")
except KeyboardInterrupt:
    INPUT_PDF = None
    print("Upload skipped.")

assert INPUT_PDF and "courtaccess_translated" not in INPUT_PDF, \
    "❌ INPUT_PDF points to a previous output — re-upload your original PDF."



📤 Upload a PDF to translate:


Saving test7.pdf to test7 (1).pdf
✅ Uploaded: /content/courtaccess_output/test7 (1).pdf


# Cell 4 - font utilities

In [ ]:
# ============================================================
# CELL 4: FONT UTILITIES
#
# get_font_code()  — maps PDF font name to PyMuPDF built-in
#                    code. Used ONLY in insert_text fallback
#                    path in Cell 9. Not used by insert_htmlbox.
#
# get_css_font()   — maps PDF font name DIRECTLY to a CSS
#                    font-family string + bold/italic flags.
#                    Used by Cell 9 _spans_to_style_groups and
#                    _build_rich_html. Avoids the lossy
#                    intermediate PyMuPDF code step:
#                      Before: name → PyMuPDF code → CSS family
#                      Now:    name → CSS family (direct)
#
# get_font_size()  — extracts original span size with a minimum
#                    floor. Original size always used, no shrinking.
#                    Overflow handled by Cell 9 _apply_vertical_push.
#
# _safe_color()    — decodes a span's packed 24-bit integer color
#                    to an (r, g, b) float tuple, bypassing the
#                    buggy sRGB_to_rgb in some PyMuPDF builds.
#
# ============================================================


def get_font_code(span):
    """
    Map original span font to closest PyMuPDF built-in font code.
    Used only in the insert_text fallback path in Cell 9.
    Bold/italic/mono/serif detected from flags and font name.
    """
    fontname = span.get("font", "")
    flags    = span.get("flags", 0)
    fn_lower = fontname.lower()

    is_bold   = (
        bool(flags & 16) or
        any(w in fn_lower for w in ["bold", "black", "heavy", "demi"])
    )
    is_italic = (
        bool(flags & 2) or
        any(w in fn_lower for w in ["italic", "oblique", "slant"])
    )
    is_mono   = (
        bool(flags & 8) or
        any(w in fn_lower for w in ["courier", "mono", "typewriter", "fixed"])
    )
    is_serif  = (
        bool(flags & 4) or
        any(w in fn_lower for w in
            ["times", "serif", "nimbus", "georgia", "garamond", "palatino"])
    )

    if is_mono:
        if is_bold and is_italic: return "cobi"
        if is_bold:               return "cobo"
        if is_italic:             return "coit"
        return "cour"
    elif is_serif:
        if is_bold and is_italic: return "tibi"
        if is_bold:               return "tibo"
        if is_italic:             return "tiit"
        return "tiro"
    else:
        if is_bold and is_italic: return "hebi"
        if is_bold:               return "hebo"
        if is_italic:             return "heit"
        return "helv"


def get_css_font(span) -> tuple:
    """
    Map original PDF span directly to CSS font properties.
    Returns (font_family_str, is_bold, is_italic).

    Used by Cell 9 _spans_to_style_groups so that style groups
    carry a CSS family string directly — no _CSS_FONTS lookup
    dict needed in Cell 9 at all.

    Priority:
      1. Monospace → 'Courier New', Courier, monospace
      2. Serif     → 'Times New Roman', Times, serif
      3. Default   → Helvetica, Arial, sans-serif
    Bold/italic inferred from both span flags and font name.
    """
    fontname = span.get("font", "")
    flags    = span.get("flags", 0)
    fn_lower = fontname.lower()

    is_bold   = (
        bool(flags & 16) or
        any(w in fn_lower for w in ["bold", "black", "heavy", "demi"])
    )
    is_italic = (
        bool(flags & 2) or
        any(w in fn_lower for w in ["italic", "oblique", "slant"])
    )
    is_mono   = (
        bool(flags & 8) or
        any(w in fn_lower for w in ["courier", "mono", "typewriter", "fixed"])
    )
    is_serif  = (
        bool(flags & 4) or
        any(w in fn_lower for w in
            ["times", "serif", "nimbus", "georgia", "garamond", "palatino"])
    )

    if is_mono:
        family = "'Courier New', Courier, monospace"
    elif is_serif:
        family = "'Times New Roman', Times, serif"
    else:
        family = "Helvetica, Arial, sans-serif"

    return family, is_bold, is_italic


def get_font_size(span, min_size=6.0) -> float:
    """
    Extract exact font size from span.
    Always returns the original size — no shrinking.
    Overflow is handled by vertical cell expansion in Cell 9.
    """
    return max(round(span.get("size", 10.0), 1), min_size)


def _safe_color(span) -> tuple:
    """
    Convert a PyMuPDF span's packed integer color to an (r, g, b)
    tuple with values in 0.0–1.0 range, as insert_text requires.

    Bypasses sRGB_to_rgb entirely — in some PyMuPDF builds it
    returns 0-255 integers instead of 0-1 floats, causing:
      ValueError: need 1, 3 or 4 color components in range 0 to 1

    The span "color" field is always a packed 24-bit integer:
      0x000000 = black, 0xFFFFFF = white, 0xFF0000 = red, etc.
    """
    if isinstance(span, dict):
        raw = span.get("color", 0)
    else:
        raw = span

    if isinstance(raw, (tuple, list)) and len(raw) >= 3:
        vals = [v / 255.0 if v > 1 else float(v) for v in raw[:3]]
        return tuple(vals)

    try:
        n = int(raw)
    except (TypeError, ValueError):
        return (0.0, 0.0, 0.0)

    r = ((n >> 16) & 0xFF) / 255.0
    g = ((n >>  8) & 0xFF) / 255.0
    b = ( n        & 0xFF) / 255.0
    return (r, g, b)


print("✅ Font utilities ready")
print("   get_font_code : PyMuPDF built-in code (insert_text fallback only)")
print("   get_css_font  : original font name → CSS directly (insert_htmlbox)")
print("   get_font_size : original size preserved (no shrinking)")
print("   _safe_color   : packed int → (r,g,b) float tuple")

✅ Font utilities ready
   get_font_code : PyMuPDF built-in code (insert_text fallback only)
   get_css_font  : original font name → CSS directly (insert_htmlbox)
   get_font_size : original size preserved (no shrinking)
   _safe_color   : packed int → (r,g,b) float tuple


# Cell 5 - page classifier

In [ ]:
# ============================================================
# CELL 5: PAGE CLASSIFIER
# DIGITAL  — real vector text and/or drawings, no background image
# SCANNED  — image-based page (with or without embedded OCR text)
# BLANK    — no text, no images, no drawings
#
# Classification signals (in priority order):
#   1. img_coverage > 0.4  → SCANNED (background image dominates)
#   2. images + no text    → SCANNED (pure scan, no OCR layer)
#   3. span_count > 5      → DIGITAL (sufficient vector text)
#   4. drawings > 10       → DIGITAL (form structure is vector-drawn
#                            even if text spans are sparse — e.g. a
#                            mostly-blank form page with boxes/lines)
#   5. fallback            → SCANNED
#
# Why drawings matter:
#   Cell 9 reads drawings via _get_cell_rects() to detect table
#   grid structure. Pages with many drawings but few spans are
#   digital forms, not scans — the old fallback misclassified them.
# ============================================================

def classify_page(page):
    td = page.get_text("dict", flags=pymupdf.TEXTFLAGS_TEXT)

    span_count = 0
    for b in td["blocks"]:
        if b["type"] == 0:
            for line in b["lines"]:
                for span in line["spans"]:
                    if span["text"].strip():
                        span_count += 1

    images    = page.get_images(full=True)
    drawings  = page.get_drawings()
    page_area = page.rect.width * page.rect.height

    # ── Compute image coverage ────────────────────────────────
    total_img_area = 0
    for img in images:
        try:
            for r in page.get_image_rects(img[0]):
                total_img_area += r.width * r.height
        except Exception:
            pass

    # Fallback: use image block bboxes if get_image_rects fails
    if total_img_area == 0:
        for b in td["blocks"]:
            if b["type"] == 1:
                bb = b["bbox"]
                total_img_area += (bb[2] - bb[0]) * (bb[3] - bb[1])

    # Fallback: if images exist but still can't measure, assume large
    if total_img_area == 0 and images:
        for img in images:
            if img[2] > 500 and img[3] > 500:
                total_img_area = page_area * 0.85
                break

    img_coverage  = total_img_area / page_area if page_area > 0 else 0
    drawing_count = len(drawings)

    def _result(page_type, is_scanned):
        return {
            "page_type":    page_type,
            "is_scanned":   is_scanned,
            "span_count":   span_count,
            "images":       len(images),
            "drawings":     drawing_count,
            "img_coverage": round(img_coverage * 100, 1),
        }

    # ── Classification logic ──────────────────────────────────

    # Blank: no text, no images, no drawings
    if span_count == 0 and not images and drawing_count == 0:
        return _result("BLANK", False)

    # Scanned: large background image dominates, or images with no text
    if img_coverage > 0.4 or (images and span_count == 0):
        return _result("SCANNED", True)

    # Digital: sufficient vector text
    if span_count > 5:
        return _result("DIGITAL", False)

    # Digital: form structure present via vector drawings even if
    # text spans are sparse (e.g. mostly-blank form page with
    # table boxes, checkbox outlines, underlines).
    # Threshold of 10 avoids triggering on pages with only a few
    # incidental path strokes.
    if drawing_count > 10:
        return _result("DIGITAL", False)

    # Fallback: too few signals to confirm digital — treat as scanned
    return _result("SCANNED", True)


def is_content_image(info):
    """
    True for pages that are pure images with no readable text.
    These are preserved as-is without OCR or translation.
    """
    return info["img_coverage"] > 80 and info["span_count"] < 5


print("✅ Page classifier ready")
print("   Signals: img_coverage → span_count → drawing_count")
print("   Drawing threshold: >10 drawings = DIGITAL form structure")

✅ Page classifier ready
   Signals: img_coverage → span_count → drawing_count
   Drawing threshold: >10 drawings = DIGITAL form structure


In [ ]:
import pymupdf

doc = pymupdf.open("test4.pdf")  # replace with your actual filename
page = doc[0]
td = page.get_text("dict")
blocks = [b for b in td["blocks"] if b["type"] == 0]
print(f"Total blocks: {len(blocks)}")
for i, b in enumerate(blocks):
    first_text = b["lines"][0]["spans"][0]["text"][:60]
    bbox = b["bbox"]
    print(f"Block {i}: bbox={bbox} | text='{first_text}'")

Total blocks: 15
Block 0: bbox=(14.399999618530273, 777.47998046875, 211.9459991455078, 785.6699829101562) | text='Standardized (BMC, DC, PFC, SC) – Civil – TC0061 (09/24) – P'
Block 1: bbox=(66.86399841308594, 12.312007904052734, 580.1429443359375, 38.08800506591797) | text='Massachusetts Trial Court: Chapter 209A Complaint Packet '
Block 2: bbox=(66.86399841308594, 39.594268798828125, 573.2227783203125, 58.15298843383789) | text='Abuse Prevention Orders (also known as “Restraining Orders” '
Block 3: bbox=(14.398999214172363, 61.838409423828125, 576.6209106445312, 89.64799499511719) | text='Under Chapter 209A of the Massachusetts General Laws, judges'
Block 4: bbox=(14.39900016784668, 95.21841430664062, 598.9085083007812, 148.20799255371094) | text='The law defines abuse as “the occurrence of any of the follo'
Block 5: bbox=(14.39900016784668, 153.77841186523438, 586.2534790039062, 182.1310272216797) | text='You can apply for an order at most courthouses (Boston Munic'
Block 6: bbox=(

# Cell 6 - NLLB translation

In [ ]:
# ============================================================
# CELL 6: NLLB-200 1.3B TRANSLATION SETUP — v3
# ============================================================
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM
import spacy
import re

MODEL_NAME  = "facebook/nllb-200-distilled-1.3B"
SOURCE_LANG = "eng_Latn"
TARGET_LANG = "spa_Latn"

print("⏳ Loading NLLB-200 1.3B...")
nllb_tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
nllb_model     = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME).to(device)
nllb_model.eval()
print("✅ NLLB-200 1.3B ready")

print("⏳ Loading spaCy NER...")
nlp = spacy.load("en_core_web_lg")
print("✅ spaCy NER ready")


# ── Citation patterns (verbatim protection) ───────────────────
CITATION_PATTERNS = [
    r"G\.L\.\s+c\.\s+[\d]+[A-Za-z]?",
    r"M\.G\.L\.\s+c\.\s+\d+",
    r"§\s*\d+[A-Za-z]*",
    r"Mass\.\s+R\.\s+\w+\.\s*P\.\s*\d+",
    r"\d+\s+U\.S\.C\.\s+§?\s*\d+",
    r"P\.\s*\d+\.\d+",
    r"(?<!\w)\d{4}-\w+-\d+(?!\w)",
    r"TC\d+\s*\(\d+\.\d+\)",
    r"TC\d+",
    r"LC-[A-Z\-]+",
    r"https?://\S+",
    r"www\.\S+",
    r"\b[A-Z]{2,}-\d+\b",
    r"FA/HA-\d+",
    r"St\.\s*\d{4},\s*c\.\s*\d+",
]

# ── Known short labels ────────────────────────────────────────
_KNOWN_LABELS = {
    "signature":                    "FIRMA",
    "dated":                        "FECHA",
    "date":                         "FECHA",
    "date signed":                  "FECHA DE FIRMA",
    "print":                        "IMPRIMIR",
    "county":                       "CONDADO",
    "yes":                          "Sí",
    "no":                           "No",
    "x":                            "X",
    "n/a":                          "N/A",
    "docket no.":                   "NÚM. DE EXPEDIENTE",
    "docket no":                    "NÚM. DE EXPEDIENTE",
    "bbo no.":                      "NÚM. DE BBO",
    "bbo no":                       "NÚM. DE BBO",
    "case no.":                     "NÚM. DE CASO",
    "case no":                      "NÚM. DE CASO",
    "page":                         "Página",
    "of":                           "de",
    "plaintiffs signature":         "FIRMA DEL DEMANDANTE",
    "signature of defendant":       "FIRMA DEL ACUSADO",
    "signature of judge":           "FIRMA DEL JUEZ",
    "signature of defense counsel": "FIRMA DEL ABOGADO DEFENSOR",
    "defendant name":               "NOMBRE DEL ACUSADO",
    "court department":             "DEPARTAMENTO DEL TRIBUNAL",
    "court division":               "DIVISIÓN DEL TRIBUNAL",
    "court division:":              "DIVISIÓN DEL TRIBUNAL:",
    "statutory excerpts":           "EXTRACTOS LEGALES",
    "judges certificate":           "CERTIFICADO DEL JUEZ",
    "printed name":                 "NOMBRE IMPRESO",
    "phone":                        "Teléfono",
    "text":                         "Mensaje de texto",
    "email":                        "Correo electrónico",
    "other:":                       "Otros:",
    "other":                        "Otros",
    "name:":                        "Nombre:",
    "age:":                         "Edad:",
    "do not know":                  "No lo sé",
    "on or about":                  "En o alrededor de",
    "as provided by plaintiff":     "SEGÚN LO PROPORCIONADO POR EL DEMANDANTE",
}

# ── NER exclusion terms ───────────────────────────────────────
_NER_EXCLUDE_WORDS = {
    "court", "department", "division", "district", "municipal",
    "superior", "juvenile", "probate", "family", "trial",
    "commonwealth", "county",
}

# ── Abbreviation expansion for glossary lookup ────────────────
_ABBREV_EXPANSIONS = {
    r'\bno\.\b':   'number',
    r'\bno\b':     'number',
    r'\bnúm\.\b':  'number',
    r'\bdept\.\b': 'department',
    r'\bdept\b':   'department',
    r'\bdiv\.\b':  'division',
    r'\bdiv\b':    'division',
    r'\batty\.\b': 'attorney',
    r'\batty\b':   'attorney',
    r'\bsig\.\b':  'signature',
    r'\bsig\b':    'signature',
}

# ── Form footer detection ─────────────────────────────────────
_FOOTER_RE = re.compile(
    r'^(?:Standardized|Standard|Estandarizado)\s*\(.*?\)\s*'
    r'(?:[-–—]\s*)?'
    r'(?:Civil|Criminal|Penal)?\s*[-–—]?\s*'
    r'[A-Z]{2,}[\-/]?\w*\s*\(\d+[./]\d+\)'
    r'(?:\s*[-–—]\s*(?:Page|Página)\s+\d+\s+(?:of|de)\s+\d+)?\s*$',
    re.IGNORECASE
)


def _is_form_footer(text: str) -> bool:
    """Detect standardized form footer lines."""
    t = text.strip()
    if _FOOTER_RE.match(t):
        return True
    if re.match(r'^(?:Standardized|Estandarizado)\s*\(', t, re.IGNORECASE):
        return True
    return False


def _translate_form_footer(text: str) -> str:
    """Minimally translate form footer — just category and page words."""
    result = text
    result = re.sub(r'\bCriminal\b',    'Penal',        result)
    result = re.sub(r'\bCivil\b',       'Civil',        result)
    result = re.sub(r'\bStandardized\b','Estandarizado', result, flags=re.IGNORECASE)
    result = re.sub(r'\bPage\b',        'Página',       result, flags=re.IGNORECASE)
    result = re.sub(r'\bof\b',          'de',           result)
    return result


def _is_blank_fill_line(text: str) -> bool:
    t = text.strip()
    if not t:
        return True
    if re.match(r'^[\s_\-]{4,}$', t):
        return True
    return len(re.sub(r'[\s_\-]', '', t)) == 0


# ── Pre-NLLB protection: citations ───────────────────────────
def _extract_citations(text: str) -> tuple:
    placeholders = {}
    protected    = text
    counter      = 0
    for pattern in CITATION_PATTERNS:
        for m in reversed(list(re.finditer(pattern, protected))):
            key               = f"RFCT{counter}RF"
            placeholders[key] = m.group(0)
            protected         = protected[:m.start()] + key + protected[m.end():]
            counter          += 1
    return protected, placeholders


# ── Pre-NLLB protection: proper nouns (NER) ──────────────────
def _extract_proper_nouns(text: str) -> tuple:
    """
    Protect proper nouns via NER placeholders.
    Excludes entities containing legal institutional words
    (court, department, etc.) — those should be translated.
    """
    doc          = nlp(text)
    placeholders = {}
    protected    = text
    counter      = 0

    for ent in sorted(doc.ents, key=lambda e: e.start_char, reverse=True):
        if ent.label_ not in ("PERSON", "GPE", "LOC", "ORG", "PRODUCT", "NORP"):
            continue
        ent_text = ent.text.strip()
        if len(ent_text) < 3:
            continue
        ent_words = {w.lower() for w in ent_text.split()}
        if ent_words & _NER_EXCLUDE_WORDS:
            continue
        if any(
            w.lower() in {"the","of","in","at","for","and","or","a","an","by","to","from"}
            for w in ent_text.split() if w[0].islower()
        ):
            continue
        key               = f"RFPN{counter}RF"
        placeholders[key] = ent_text
        protected         = protected[:ent.start_char] + key + protected[ent.end_char:]
        counter          += 1
    return protected, placeholders


def _restore_placeholders(text: str, *dicts) -> str:
    """
    Restore RFCT/RFPN placeholders back to original text.
    Handles spaces, case changes, and punctuation NLLB may insert.
    """
    result = text
    for d in dicts:
        for key, original in d.items():
            if key in result:
                result = result.replace(key, original)
                continue
            num = re.search(r'\d+', key)
            if num:
                prefix  = key[:key.index(num.group())]
                suffix  = key[key.index(num.group()) + len(num.group()):]
                pattern = (
                    r'[,;:\s]*'
                    + re.escape(prefix) + r'\s*'
                    + num.group() + r'\s*'
                    + re.escape(suffix)
                    + r'[,;:\s]*'
                )
                match = re.search(pattern, result, flags=re.IGNORECASE)
                if match:
                    result = result[:match.start()] + original + result[match.end():]
    result = re.sub(r'RF[A-Z]{2,4}\d+RF', '', result)
    return result.strip()


def _is_preserve_only(text: str) -> bool:
    stripped = text.strip()
    if not stripped:
        return True
    cleaned = re.sub(r"RF[A-Z]+\d+RF", "", stripped)
    cleaned = re.sub(r"[\d\s\.\-\/\(\)\,\;\:\#\@\!\?]+", "", cleaned)
    return len(cleaned.strip()) == 0


# ── Glossary enforcement (with variant matching) ──────────────
def _expand_abbreviations(text: str) -> str:
    result = text.lower()
    for pattern, expansion in _ABBREV_EXPANSIONS.items():
        result = re.sub(pattern, expansion, result, flags=re.IGNORECASE)
    return result


def _apply_glossary_corrections(original: str, translated: str) -> str:
    """
    Single-pass glossary enforcement with variant matching.
    Context-dependent terms (CONTEXT_DEPENDENT_ES) are skipped.
    """
    try:
        glossary = GLOSSARY_ES
    except NameError:
        return translated

    try:
        skip_terms = {k.lower() for k in CONTEXT_DEPENDENT_ES.keys()}
    except NameError:
        skip_terms = set()

    result        = translated
    orig_lower    = original.lower()
    orig_expanded = _expand_abbreviations(original)

    for en_term in sorted(glossary.keys(), key=len, reverse=True):
        if en_term not in orig_lower and en_term not in orig_expanded:
            continue
        if en_term in skip_terms:
            continue

        correct_es   = glossary[en_term]
        alternatives = [a.strip() for a in correct_es.split(",") if a.strip()]
        if not alternatives:
            continue

        already_correct = False
        for alt in alternatives:
            stem = alt[:6].lower() if len(alt) >= 6 else alt.lower()
            if stem in result.lower():
                already_correct = True
                break
        if already_correct:
            continue

        en_pattern = r'(?<!\w)' + re.escape(en_term) + r'(?!\w)'
        if re.search(en_pattern, result, re.IGNORECASE):
            result = re.sub(en_pattern, alternatives[0], result, flags=re.IGNORECASE)
            continue

        for abbr_pat, expansion in _ABBREV_EXPANSIONS.items():
            if expansion in en_term:
                abbr_form    = en_term.replace(expansion, abbr_pat.strip('\\b'))
                abbr_form    = abbr_form.replace('\\b', '').replace('\\', '')
                abbr_pattern = r'(?<!\w)' + re.escape(abbr_form) + r'(?!\w)'
                if re.search(abbr_pattern, result, re.IGNORECASE):
                    result = re.sub(abbr_pattern, alternatives[0], result, flags=re.IGNORECASE)
                    break

    return result


# ── Core NLLB translation ─────────────────────────────────────
def _raw_batch_translate(texts: list) -> list:
    nllb_tokenizer.src_lang = SOURCE_LANG
    enc = nllb_tokenizer(
        texts, return_tensors="pt", padding=True,
        truncation=True, max_length=512
    ).to(device)
    gen = nllb_model.generate(
        **enc,
        forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids(TARGET_LANG),
        max_length=min(512, enc["input_ids"].shape[1] + 60),
        num_beams=4,
        early_stopping=True,
    )
    return nllb_tokenizer.batch_decode(gen, skip_special_tokens=True)


# ── Main translate_one ────────────────────────────────────────
def translate_one(text: str) -> str:
    if not text or not text.strip():
        return text

    # Step 0: preserve blank fill-in lines
    if _is_blank_fill_line(text):
        return text

    # Step 0b: known short labels — bypass NLLB entirely
    label_key = text.strip().lower()
    if label_key in _KNOWN_LABELS:
        return _KNOWN_LABELS[label_key]

    # Step 0b2: known labels after stripping punctuation
    label_key_stripped = re.sub(r'[^\w\s]', '', label_key).strip()
    if label_key_stripped in _KNOWN_LABELS:
        return _KNOWN_LABELS[label_key_stripped]

    # Step 0c: very short text (≤ 2 non-space chars) — preserve as-is
    stripped_alpha = re.sub(r'[\s\-_\.]+', '', text)
    if len(stripped_alpha) <= 2:
        return text

    # Step 0d: form footers — minimal translation, no NLLB
    if _is_form_footer(text):
        return _translate_form_footer(text)

    # Step 0e: text that is ONLY underscores/fill — preserve as-is
    if re.match(r'^[\s_\-\.]+$', text.strip()):
        return text

    # Step 1: protect citations → RFCT placeholders
    protected, cite_map = _extract_citations(text)

    # Step 2: protect proper nouns → RFPN placeholders
    protected, prop_map = _extract_proper_nouns(protected)

    # Step 3: nothing real left to translate
    if _is_preserve_only(protected):
        return _restore_placeholders(text, cite_map, prop_map)

    # Step 4: NLLB translates
    try:
        translated_raw = _raw_batch_translate([protected])[0]
    except Exception as e:
        print(f"   ⚠️  NLLB error: {e} — keeping original")
        return text

    # Step 5: restore placeholders
    translated = _restore_placeholders(translated_raw, cite_map, prop_map)

    # ── TRACE hooks (no-op when TRACE_ENABLED is False) ───────
    trace_nllb_raw(translated)
    trace_glossary_analysis(text, translated)

    # Step 6: glossary enforcement
    translated = _apply_glossary_corrections(text, translated)

    # ── TRACE hook ────────────────────────────────────────────
    trace_post_glossary(translated)

    # Step 7: hallucination guard
    ratio = len(translated) / max(len(text), 1)
    if ratio > 2.5 or ratio < 0.1:
        if label_key in _KNOWN_LABELS:
            return _KNOWN_LABELS[label_key]
        print(f"   ⚠️  Hallucination (ratio={ratio:.1f}): '{text[:50]}' → keeping original")
        return text

    # Step 8: deduplicate repeated clauses
    parts  = [p.strip() for p in translated.split(',')]
    deduped = [parts[0]]
    for part in parts[1:]:
        if part.lower() != deduped[-1].lower():
            deduped.append(part)
    translated = ', '.join(deduped)

    return translated


def batch_translate(texts: list) -> list:
    return [translate_one(t) for t in texts]


def split_into_sentences(text: str) -> list:
    sentences = re.split(r"(?<=[.!?])\s+", text.strip())
    return [s for s in sentences if s.strip()]


print("✅ Translation pipeline v3 ready")
print("   Short label bypass: DATE, SIGNATURE, X, BBO NO., DOCKET NO., etc.")
print("   NER exclusion: court/department/division terms flow to NLLB")
print("   Glossary variant matching: 'docket no.' → 'docket number'")
print("   Form footer detection: minimal translation, no NLLB")
print("   Steps: labels → footer → citations → NER → NLLB → glossary → guard")

⏳ Loading NLLB-200 1.3B...
✅ NLLB-200 1.3B ready
⏳ Loading spaCy NER...
✅ spaCy NER ready
✅ Translation pipeline v3 ready
   Short label bypass: DATE, SIGNATURE, X, BBO NO., DOCKET NO., etc.
   NER exclusion: court/department/division terms flow to NLLB
   Glossary variant matching: 'docket no.' → 'docket number'
   Form footer detection: minimal translation, no NLLB
   Steps: labels → footer → citations → NER → NLLB → glossary → guard


# Cell 7a - Legal Glossary

In [ ]:
# ============================================================
# CELL 7A: LEGAL GLOSSARY LOADER (OPTIMIZED)
#
# The glossary is the SOURCE OF TRUTH for legal terminology.
# When NLLB's output disagrees with the glossary, glossary wins.
#
# Two tiers of terms:
#   STRICT  — Always enforced. Single unambiguous translation.
#             e.g., child → menor, affidavit → declaración jurada
#   REFERENCE — Injected into Llama prompt as guidance but NOT
#             auto-replaced, because context determines the
#             correct choice.
#             e.g., defendant → acusado OR demandado (depends on
#             civil vs criminal context)
#
# MA_OVERRIDES updates the NJ glossary with MA-specific terms.
# CONTEXT_DEPENDENT terms are excluded from auto-correction
# but still provided to Llama as reference.
# ============================================================
import urllib.request
import json
import re
import os

SPANISH_GLOSSARY_URL  = "https://www.njcourts.gov/sites/default/files/forms/11783_glossary_spanish.pdf"
SPANISH_GLOSSARY_PDF  = os.path.join(WORK_DIR, "glossary_spanish.pdf")
SPANISH_GLOSSARY_JSON = os.path.join(WORK_DIR, "glossary_es.json")

# ── MA-specific overrides (STRICT tier) ───────────────────────
# These have ONE correct translation in MA legal context.
# They override NJ glossary entries AND get auto-enforced
# in Cell 6's glossary correction pass.
MA_OVERRIDES_ES = {
    # Institutional names
    "commonwealth":              "Commonwealth",
    "trial court":               "Tribunal de Justicia",
    "municipal court":           "Tribunal Municipal",

    # Legal citations (preserved verbatim)
    "g.l. c.":                   "G.L. c.",

    # Unambiguous legal terms
    "beyond a reasonable doubt": "más allá de una duda razonable",
    "mandatory minimum":         "mínimo obligatorio",
    "waiver":                    "renuncia",
    "waive":                     "renunciar",
    "plea":                      "declaración",
    "juror":                     "miembro del jurado",
    "right to counsel":          "derecho a la asistencia letrada",
    "due process":               "debido proceso",
    "contempt":                  "desacato",
    "public defender":           "defensor público",
    "verdict":                   "veredicto",
    "party":                     "parte",
    "parties":                   "partes",
    "notice":                    "aviso",
    "partition":                 "partición",

    # Child terminology — always "menor" in MA court documents
    "child":                     "menor",
    "children":                  "menores",
    "child(ren)":                "menor(es)",
    "child's":                   "del menor",
    "the child":                 "el menor",
    "the children":              "los menores",
}

# ── Context-dependent terms (REFERENCE tier) ──────────────────
# These have MULTIPLE valid translations depending on case type.
# NOT auto-enforced — only injected into Llama prompt.
# Llama uses document context to choose the right one.
CONTEXT_DEPENDENT_ES = {
    "defendant":  "acusado (criminal) / demandado (civil)",
    "plaintiff":  "demandante / accionante",
    "counsel":    "abogado / asesor legal",
    "court":      "tribunal / juzgado",
    "order":      "orden / mandato / fallo",
    "motion":     "moción / petición",
    "hearing":    "audiencia / vista",
    "judgment":   "sentencia / fallo",
    "appeal":     "apelación / recurso",
    "custody":    "custodia (children) / posesión (property)",
}


# Lines to skip during glossary PDF parsing
SKIP_LINES = {
    "glossary of legal",
    "page ",
    "página",
    "cn 11783",
    "a , b , c",
}


def _ensure_glossary_pdf() -> bool:
    if os.path.exists(SPANISH_GLOSSARY_PDF):
        return True

    print(f"   ⏳ Attempting download from NJ Courts...")
    try:
        req = urllib.request.Request(
            SPANISH_GLOSSARY_URL,
            headers={"User-Agent": "Mozilla/5.0"}
        )
        with urllib.request.urlopen(req, timeout=30) as resp, \
             open(SPANISH_GLOSSARY_PDF, "wb") as f:
            f.write(resp.read())
        size_kb = os.path.getsize(SPANISH_GLOSSARY_PDF) / 1024
        print(f"   ✅ Downloaded: {size_kb:.0f} KB")
        return True
    except Exception as e:
        print(f"   ⚠️  Auto-download failed ({e})")

    print("\n   📤 Please upload the glossary PDF manually.")
    print("   Download it from:")
    print(f"   {SPANISH_GLOSSARY_URL}")
    print("   Then upload it in the file picker below.\n")
    try:
        from google.colab import files
        uploaded = files.upload()
        if uploaded:
            fname = list(uploaded.keys())[0]
            import shutil
            shutil.copy(fname, SPANISH_GLOSSARY_PDF)
            size_kb = os.path.getsize(SPANISH_GLOSSARY_PDF) / 1024
            print(f"   ✅ Saved: {size_kb:.0f} KB → {SPANISH_GLOSSARY_PDF}")
            return True
        else:
            print("   ❌ No file uploaded.")
            return False
    except Exception as e:
        print(f"   ❌ Upload failed: {e}")
        return False


def _should_skip(text: str, y0: float) -> bool:
    if y0 < 55 or y0 > 725:
        return True
    t = text.lower().strip()
    if not t:
        return True
    if len(t) <= 2 and t.isalpha():
        return True
    if any(t.startswith(skip) for skip in SKIP_LINES):
        return True
    return False


def parse_spanish_glossary(pdf_path: str) -> dict:
    doc      = pymupdf.open(pdf_path)
    glossary = {}

    for pg_num in range(2, len(doc)):
        page = doc[pg_num]
        td   = page.get_text("dict")

        for b in td["blocks"]:
            if b["type"] != 0:
                continue
            b_y0 = round(b["bbox"][1], 1)
            if b_y0 < 55 or b_y0 > 725:
                continue

            lines = []
            for line in b["lines"]:
                text = " ".join(s["text"] for s in line["spans"]).strip()
                if text:
                    lines.append(text)
            if not lines or _should_skip(lines[0], b_y0):
                continue

            if len(lines) == 1:
                continue
            elif len(lines) == 2:
                en, es = lines[0].strip(), lines[1].strip()
                if en and es:
                    glossary[en.strip(" ,;").lower()] = es
            else:
                line_x0s = [
                    round(l["bbox"][0], 1) for l in b["lines"]
                    if " ".join(s["text"] for s in l["spans"]).strip()
                ]
                left_lines, right_lines = [], []
                for line, x0 in zip(
                    [l for l in b["lines"]
                     if " ".join(s["text"] for s in l["spans"]).strip()],
                    line_x0s
                ):
                    text = " ".join(s["text"] for s in line["spans"]).strip()
                    (left_lines if x0 < 200 else right_lines).append(text)

                if left_lines and right_lines:
                    en = " ".join(left_lines).strip()
                    es = " ".join(right_lines).strip()
                else:
                    en = left_lines[0] if left_lines else lines[0]
                    es = " ".join(
                        (left_lines[1:] if left_lines else lines[1:])
                    ).strip()
                if en and es:
                    glossary[en.strip(" ,;").lower()] = es

    doc.close()

    # Apply MA overrides (these take priority over NJ glossary)
    glossary.update({k.lower(): v for k, v in MA_OVERRIDES_ES.items()})

    print(f"   📊 Parsed {len(glossary)} term pairs from PDF")
    return glossary


def load_glossary_es(force_reparse: bool = False) -> dict:
    os.makedirs(WORK_DIR, exist_ok=True)

    if not force_reparse and os.path.exists(SPANISH_GLOSSARY_JSON):
        with open(SPANISH_GLOSSARY_JSON, "r", encoding="utf-8") as f:
            glossary = json.load(f)
        print(f"   📚 Loaded from cache: {len(glossary)} terms")
        return glossary

    if not _ensure_glossary_pdf():
        return {k.lower(): v for k, v in MA_OVERRIDES_ES.items()}

    print("   ⏳ Parsing glossary PDF...")
    glossary = parse_spanish_glossary(SPANISH_GLOSSARY_PDF)

    with open(SPANISH_GLOSSARY_JSON, "w", encoding="utf-8") as f:
        json.dump(glossary, f, ensure_ascii=False, indent=2)
    print(f"   ✅ Cached to JSON: {SPANISH_GLOSSARY_JSON}")

    return glossary


def get_matching_glossary_terms(text: str, glossary: dict,
                                max_terms: int = 8) -> dict:
    """Get glossary terms that appear in the text (for Llama reference)."""
    text_lower = text.lower()
    matches    = {}
    for en_term in sorted(glossary.keys(), key=len, reverse=True):
        if en_term in text_lower:
            matches[en_term] = glossary[en_term]
        if len(matches) >= max_terms:
            break
    return matches


def get_context_dependent_terms(text: str, max_terms: int = 5) -> dict:
    """Get context-dependent terms for Llama reference prompt."""
    text_lower = text.lower()
    matches    = {}
    for en_term in sorted(CONTEXT_DEPENDENT_ES.keys(), key=len,
                          reverse=True):
        if en_term in text_lower:
            matches[en_term] = CONTEXT_DEPENDENT_ES[en_term]
        if len(matches) >= max_terms:
            break
    return matches


# ── Load ──────────────────────────────────────────────────────
print("⏳ Loading Spanish legal glossary...")
GLOSSARY_ES = load_glossary_es(force_reparse=False)
print(f"✅ Spanish glossary ready: {len(GLOSSARY_ES)} terms")
print(f"   Strict terms (auto-enforced): {len(MA_OVERRIDES_ES)}")
print(f"   Context-dependent (Llama only): {len(CONTEXT_DEPENDENT_ES)}")

# ── Sanity check ──────────────────────────────────────────────
print("\n📋 Sanity check — key terms:")
test_terms = [
    "arraignment", "bail", "defendant (civil)",
    "defendant (criminal)", "waiver", "affidavit",
    "bench warrant", "beyond a reasonable doubt",
    "plea", "probation", "trial court", "municipal court",
    "child",
]
for t in test_terms:
    es     = GLOSSARY_ES.get(t, "NOT FOUND")
    status = "✅" if es != "NOT FOUND" else "❌"
    print(f"   {status} {t:40s} → {es}")

found = sum(1 for t in test_terms if GLOSSARY_ES.get(t) != "NOT FOUND")
print(f"\n   {found}/{len(test_terms)} test terms found")
print(f"   Total glossary size: {len(GLOSSARY_ES)} terms")

⏳ Loading Spanish legal glossary...
   📚 Loaded from cache: 741 terms
✅ Spanish glossary ready: 741 terms
   Strict terms (auto-enforced): 25
   Context-dependent (Llama only): 10

📋 Sanity check — key terms:
   ✅ arraignment                              → lectura de cargos, instrucción de cargos
   ✅ bail                                     → fianza, caución
   ✅ defendant (civil)                        → demandado (en una acción civil)
   ✅ defendant (criminal)                     → acusado (en una causa penal)
   ✅ waiver                                   → renuncia
   ✅ affidavit                                → Affidávit, afidávit, declaración jurada (por escrito)
   ✅ bench warrant                            → orden de busca y captura/ de detención (emitida verbalmente por el juez debido a incomparecencia)
   ✅ beyond a reasonable doubt                → más allá de una duda razonable
   ✅ plea                                     → declaración
   ✅ probation                       

# Cell 7 - Llama 4 context verification

In [ ]:
# ============================================================
# CELL 7: VERTEX AI LLAMA 4 LEGAL VERIFICATION
#
# Llama is the FINAL verification layer. It receives:
#   - STRICT glossary terms (source of truth, already enforced
#     by Cell 6 where possible — Llama catches remaining gaps)
#   - CONTEXT-DEPENDENT terms with alternatives (Llama chooses
#     the right variant based on document context)
#   - The EN/ES pair to verify
#
# Llama's job:
#   1. Fix context-dependent terms Cell 6 couldn't handle
#   2. Catch meaning errors NLLB introduced
#   3. Verify glossary enforcement didn't break grammar
#   4. Leave correct translations unchanged
# ============================================================
import json

VERIFICATION_MODE = "document"  # "document" or "audio"


def _strip_fences(text: str) -> str:
    text = text.strip()
    for prefix in ("```json", "```"):
        if text.startswith(prefix):
            text = text[len(prefix):]
    if text.endswith("```"):
        text = text[:-3]
    return text.strip()


def _build_glossary_snippet(texts: list) -> str:
    """
    Build reference snippet for Llama prompt containing:
      1. Strict glossary terms found in this batch
      2. Context-dependent terms found in this batch (with alternatives)

    Returns formatted string for prompt injection.
    """
    combined = " ".join(texts).lower()

    # Strict terms from glossary
    strict = {}
    for en_term in sorted(GLOSSARY_ES.keys(), key=len, reverse=True):
        if en_term in combined:
            strict[en_term] = GLOSSARY_ES[en_term]
        if len(strict) >= 12:
            break

    # Context-dependent terms
    context = {}
    try:
        for en_term in sorted(CONTEXT_DEPENDENT_ES.keys(), key=len,
                              reverse=True):
            if en_term in combined:
                context[en_term] = CONTEXT_DEPENDENT_ES[en_term]
            if len(context) >= 6:
                break
    except NameError:
        pass

    parts = []
    if strict:
        parts.append("VERIFIED TERMS (use these):")
        parts.extend(f"  {en} → {es}" for en, es in strict.items())
    if context:
        parts.append("CONTEXT-DEPENDENT (choose based on document type):")
        parts.extend(f"  {en} → {es}" for en, es in context.items())

    return "\n".join(parts) if parts else ""


def _call_llama(
    original_batch: list,
    translated_batch: list,
    glossary_snippet: str,
    _retry: bool = False
) -> list:
    pairs = "\n".join(
        f"{k+1}. EN: {o}\n   ES: {t}"
        for k, (o, t) in enumerate(zip(original_batch, translated_batch))
    )

    glossary_section = ""
    if glossary_snippet:
        glossary_section = (
            "\nREFERENCE GLOSSARY:\n"
            f"{glossary_snippet}\n"
        )

    prompt = (
        "You are a certified legal translator specializing in "
        "English to Spanish court document translation.\n"
        f"{glossary_section}\n"
        "For each numbered pair below, verify the Spanish "
        "translation is legally accurate.\n\n"
        "Rules:\n"
        "1. VERIFIED TERMS in the glossary are the source of truth. "
        "If the translation uses a different word for a verified "
        "term, correct it to match the glossary.\n"
        "2. CONTEXT-DEPENDENT terms have multiple valid translations. "
        "Choose the right one based on document context "
        "(e.g., 'acusado' for criminal, 'demandado' for civil).\n"
        "3. Preserve legal citations, § symbols, case numbers, "
        "statute references, URLs, and form codes EXACTLY as-is.\n"
        "4. Preserve proper nouns (people, places, organizations) "
        "exactly as written.\n"
        "5. Only correct genuine legal meaning errors — do NOT "
        "change translations for stylistic preference alone.\n"
        "6. If a translation is already legally accurate, "
        "return it UNCHANGED.\n"
        "7. Short field labels (DATE, SIGNATURE, BBO NO.) "
        "should be translated as standard Spanish form labels.\n"
        "8. Do NOT capitalize common nouns mid-sentence. Only "
        "capitalize proper nouns and sentence starts.\n"
        "9. In date range fields ('From ___ to ___'), use "
        "'Desde ___ hasta ___'. "
        "For 'to present' use 'hasta la actualidad'.\n"
        "Return ONLY a JSON array of verified Spanish strings, "
        "one per pair, in the same order. "
        "No explanation, no markdown, just the JSON array.\n\n"
        f"PAIRS:\n{pairs}\n\n"
        'Return format: ["verified 1", "verified 2", ...]'
    )

    try:
        resp = vertex_client.chat.completions.create(
            model=VERTEX_MODEL_ID,
            messages=[{"role": "user", "content": prompt}],
            temperature=0.1,
            max_tokens=2048,
        )
        corrected = json.loads(
            _strip_fences(resp.choices[0].message.content)
        )

        if isinstance(corrected, list) \
                and len(corrected) == len(original_batch):
            validated = []
            for orig, trans, corr in zip(
                original_batch, translated_batch, corrected
            ):
                if not corr or not corr.strip():
                    validated.append(trans)
                    continue
                ratio = len(corr) / max(len(trans), 1)
                if 0.2 <= ratio <= 5.0:
                    validated.append(corr.strip())
                else:
                    validated.append(trans)
            return validated
        else:
            print("   ⚠️  Llama length mismatch — keeping NLLB output")
            return translated_batch

    except Exception as e:
        err_str = str(e)
        if (
            not _retry
            and ("401" in err_str
                 or "UNAUTHENTICATED" in err_str
                 or "ACCESS_TOKEN_EXPIRED" in err_str)
        ):
            print("   🔄 Token expired — refreshing credentials...")
            try:
                refresh_vertex_credentials()
                return _call_llama(
                    original_batch,
                    translated_batch,
                    glossary_snippet,
                    _retry=True
                )
            except Exception as e2:
                print(f"   ⚠️  Retry failed: {e2} — keeping NLLB output")
        else:
            print(f"   ⚠️  Llama error: {e} — keeping NLLB output")
        return translated_batch


def verify_page_translations(
    original_spans: list,
    translated_spans: list,
    batch_size: int = 16
) -> list:
    if not original_spans:
        return translated_spans

    verified = list(translated_spans)

    if VERIFICATION_MODE == "document":
        indices_to_verify = list(range(len(original_spans)))
        print(f"   📋 Document mode: verifying all "
              f"{len(indices_to_verify)} spans")
    else:
        indices_to_verify = []
        for i, orig in enumerate(original_spans):
            if get_matching_glossary_terms(orig, GLOSSARY_ES):
                indices_to_verify.append(i)

        if not indices_to_verify:
            print("   ✅ Audio mode: no legal terms — skipping Llama")
            return verified

        print(f"   🎙️  Audio mode: {len(indices_to_verify)}/"
              f"{len(original_spans)} spans contain legal terms")

    total_batches = (
        len(indices_to_verify) + batch_size - 1
    ) // batch_size

    for batch_num, start in enumerate(
        range(0, len(indices_to_verify), batch_size), 1
    ):
        batch_idx   = indices_to_verify[start:start + batch_size]
        orig_batch  = [original_spans[i]   for i in batch_idx]
        trans_batch = [translated_spans[i] for i in batch_idx]

        glossary_snippet = _build_glossary_snippet(orig_batch)
        n_glossary       = (len(glossary_snippet.splitlines())
                            if glossary_snippet else 0)

        results = _call_llama(orig_batch, trans_batch, glossary_snippet)

        for idx, result in zip(batch_idx, results):
            verified[idx] = result

        print(f"   ✅ Batch {batch_num}/{total_batches}: "
              f"verified {len(batch_idx)} spans "
              f"({n_glossary} glossary terms injected)")

    return verified


print("✅ Vertex AI Llama 4 legal verification ready")
print(f"   Mode: {VERIFICATION_MODE.upper()}")
print(f"   Batch size: {16 if VERIFICATION_MODE == 'document' else 8}")
print(f"   Glossary: strict terms (enforced) + context-dependent (Llama chooses)")

✅ Vertex AI Llama 4 legal verification ready
   Mode: DOCUMENT
   Batch size: 16
   Glossary: strict terms (enforced) + context-dependent (Llama chooses)


# Llama tests

These testing cells were defined for previous experiments and are not currently in use. They can be used to test the Llama 4 legal verification system in the future, with some modifications.

In [ ]:
# ============================================================
# TEMP TEST: RTT + BERTScore accuracy comparison
# NLLB alone vs NLLB + Llama 4 verified
# ============================================================
!pip install -q bert-score
from bert_score import score as bert_score
import pymupdf

LANG_SHORT = "ES"   # label for printout

# ── Step 1: Extract text ──────────────────────────────────────
print("📄 Extracting text from uploaded PDF...")
doc = pymupdf.open(INPUT_PDF)
all_units = []
for pg in range(min(3, len(doc))):
    page = doc[pg]
    info = classify_page(page)
    if info["page_type"] == "DIGITAL":
        # v8: use _get_block_units per block
        td = page.get_text("dict", flags=pymupdf.TEXTFLAGS_TEXT)
        for blk in td["blocks"]:
            if blk["type"] == 0:
                all_units.extend(_get_block_units(blk))
doc.close()

original_texts = [
    u["text"] for u in all_units
    if u["text"].strip() and not u.get("preserve", False)
][:50]

if not original_texts:
    print("❌ No translatable text found — check your PDF")
else:
    print(f"   ✅ Extracted {len(original_texts)} translatable spans")

    # ── Step 2: NLLB only ────────────────────────────────────
    print("\n⏳ Step 2: NLLB translation (no Llama)...")
    nllb_spanish = batch_translate(original_texts)
    print(f"   ✅ NLLB translated {len(nllb_spanish)} spans")

    # ── Step 3: NLLB + Llama ─────────────────────────────────
    print("\n⏳ Step 3: NLLB + Llama 4 verification...")
    verified_spanish = verify_page_translations(
        original_texts, nllb_spanish, batch_size=8
    )
    print(f"   ✅ Verified {len(verified_spanish)} spans")

    # ── Step 4: Round-trip ───────────────────────────────────
    print("\n⏳ Step 4: Round-tripping both back to English...")

    def back_translate(texts):
        results = []
        for text in texts:
            if not text.strip():
                results.append(text)
                continue
            nllb_tokenizer.src_lang = "spa_Latn"
            enc = nllb_tokenizer(
                text, return_tensors="pt",
                truncation=True, max_length=512
            ).to(device)
            gen = nllb_model.generate(
                **enc,
                forced_bos_token_id=nllb_tokenizer.convert_tokens_to_ids(
                    "eng_Latn"
                ),
                max_length=512, num_beams=4, early_stopping=True,
            )
            results.append(
                nllb_tokenizer.batch_decode(gen, skip_special_tokens=True)[0]
            )
        nllb_tokenizer.src_lang = SOURCE_LANG   # restore
        return results

    nllb_back     = back_translate(nllb_spanish)
    verified_back = back_translate(verified_spanish)
    nllb_tokenizer.src_lang = SOURCE_LANG
    print("   ✅ Round-trip complete")

    # ── Step 5: BERTScore ────────────────────────────────────
    print("\n⏳ Step 5: Computing BERTScore...")
    P1, R1, F1_nllb     = bert_score(nllb_back,     original_texts,
                                     lang="en", verbose=False)
    P2, R2, F1_verified = bert_score(verified_back, original_texts,
                                     lang="en", verbose=False)

    nllb_mean     = F1_nllb.mean().item()
    verified_mean = F1_verified.mean().item()
    improvement   = (verified_mean - nllb_mean) * 100

    # ── Step 6: Summary ──────────────────────────────────────
    print("\n" + "="*60)
    print("📊 ACCURACY RESULTS (BERTScore F1)")
    print("="*60)
    print(f"   NLLB alone:            {nllb_mean:.4f}  ({nllb_mean*100:.1f}%)")
    print(f"   NLLB + Llama verified: {verified_mean:.4f}  ({verified_mean*100:.1f}%)")
    print(f"   Improvement:           {improvement:+.2f}%")
    print("="*60)

    # ── Step 7: Top 5 where Llama helped most ────────────────
    diffs = sorted(
        [(i, F1_verified[i].item() - F1_nllb[i].item())
         for i in range(len(original_texts))],
        key=lambda x: x[1], reverse=True
    )
    print("\n📋 5 SPANS WHERE LLAMA MADE THE BIGGEST DIFFERENCE:")
    for rank, (i, diff) in enumerate(diffs[:5], 1):
        print(f"\n  [{rank}] Score change: {diff:+.4f}")
        print(f"  EN original:    {original_texts[i][:80]}")
        print(f"  NLLB {LANG_SHORT}:       {nllb_spanish[i][:80]}")
        print(f"  Verified {LANG_SHORT}:   {verified_spanish[i][:80]}")
        print(f"  RTT (NLLB):     {nllb_back[i][:80]}")
        print(f"  RTT (Verified): {verified_back[i][:80]}")

    # ── Step 8: Spans where Llama made things worse ───────────
    worse = [(i, d) for i, d in diffs if d < -0.01]
    if worse:
        print(f"\n⚠️  {len(worse)} spans where verification scored lower:")
        for i, diff in worse[:5]:
            print(f"\n  Score change: {diff:+.4f}")
            print(f"  EN:             {original_texts[i][:180]}")
            print(f"  NLLB {LANG_SHORT}:       {nllb_spanish[i][:180]}")
            print(f"  Verified {LANG_SHORT}:   {verified_spanish[i][:180]}")
    else:
        print("\n✅ Llama verification did not degrade any spans")

📄 Extracting text from uploaded PDF...


KeyError: 'text'

In [ ]:
# ============================================================
# TERM-LEVEL ACCURACY TEST
# Depends on: original_texts, nllb_spanish, verified_spanish
# from the RTT test cell (run that first).
# ============================================================
import re

# Civil/criminal accepted alternatives for ambiguous terms
_CIVIL_OVERRIDES = {
    "defendant": ["acusado", "demandado"],
    "plaintiff":  ["demandante", "accionante"],
}

def _extract_citations_test(text: str) -> list:
    """Extract legal citations — renamed to avoid overwriting Cell 6's version."""
    patterns = [
        r"G\.L\.\s+c\.\s+[\d]+",
        r"§\s*\d+[A-Za-z]*",
        r"\d+\s+U\.S\.C\.",
        r"Mass\.\s+R\.\s+\w+\.",
        r"P\.\s*\d+\.\d+",
    ]
    found = []
    for p in patterns:
        found.extend(re.findall(p, text, re.IGNORECASE))
    return found


def _check_citation_preserved(citation: str, translated: str) -> bool:
    return citation.strip() in translated


def _check_term_translated(en_term: str, expected_es: str,
                            translated: str) -> tuple:
    translated_lower = translated.lower()
    expected_lower   = expected_es.lower()

    # Merge glossary alternatives with civil/criminal overrides
    alternatives = [t.strip() for t in expected_lower.split(",")]
    if en_term.lower() in _CIVIL_OVERRIDES:
        alternatives += _CIVIL_OVERRIDES[en_term.lower()]

    correct = False
    for alt in alternatives:
        if not alt:
            continue
        if alt in translated_lower:
            correct = True; break
        stem = alt[:6] if len(alt) >= 6 else alt
        if stem in translated_lower:
            correct = True; break

    left_untranslated = en_term.lower() in translated_lower
    return correct, left_untranslated


def run_term_level_accuracy(original_texts, nllb_translations,
                             verified_translations):
    print("=" * 65)
    print("📊 TERM-LEVEL ACCURACY REPORT")
    print("=" * 65)

    # ── 1. Glossary term accuracy ─────────────────────────────
    print("\n📚 SECTION 1: Glossary Term Translation Accuracy")
    print("-" * 65)

    nllb_correct = verif_correct = total_terms = 0
    term_results = []

    for orig, nllb, verif in zip(original_texts, nllb_translations,
                                  verified_translations):
        matches = get_matching_glossary_terms(orig, GLOSSARY_ES)
        if not matches:
            continue
        for en_term, expected_es in matches.items():
            total_terms += 1
            nllb_ok,  _ = _check_term_translated(en_term, expected_es, nllb)
            verif_ok, _ = _check_term_translated(en_term, expected_es, verif)
            if nllb_ok:  nllb_correct  += 1
            if verif_ok: verif_correct += 1
            if nllb_ok != verif_ok or (not nllb_ok and not verif_ok):
                term_results.append({
                    "term": en_term, "expected": expected_es[:40],
                    "nllb": nllb_ok, "verif": verif_ok,
                    "context": orig[:50],
                })

    if total_terms > 0:
        nllb_pct  = nllb_correct  / total_terms * 100
        verif_pct = verif_correct / total_terms * 100
        print(f"   NLLB alone:            {nllb_correct}/{total_terms} ({nllb_pct:.1f}%)")
        print(f"   NLLB + Llama verified: {verif_correct}/{total_terms} ({verif_pct:.1f}%)")
        print(f"   Improvement:           {verif_pct - nllb_pct:+.1f}%")
        if term_results:
            print(f"\n   Terms with differences:")
            for r in term_results[:10]:
                print(f"\n   Term:     '{r['term']}'")
                print(f"   Expected: '{r['expected']}'")
                print(f"   NLLB:     {'✅' if r['nllb'] else '❌'}  "
                      f"Verified: {'✅' if r['verif'] else '❌'}")
                print(f"   Context:  '{r['context']}'")
    else:
        print("   No glossary terms found in test spans")

    # ── 2. Legal citation preservation ───────────────────────
    print(f"\n\n⚖️  SECTION 2: Legal Citation Preservation")
    print("-" * 65)

    cite_total = nllb_cite_ok = verif_cite_ok = 0
    cite_results = []

    for orig, nllb, verif in zip(original_texts, nllb_translations,
                                  verified_translations):
        for cite in _extract_citations_test(orig):
            cite_total += 1
            nllb_ok  = _check_citation_preserved(cite, nllb)
            verif_ok = _check_citation_preserved(cite, verif)
            if nllb_ok:  nllb_cite_ok  += 1
            if verif_ok: verif_cite_ok += 1
            if not nllb_ok or not verif_ok:
                cite_results.append({
                    "citation": cite,
                    "nllb_ok": nllb_ok, "verif_ok": verif_ok,
                    "nllb_out": nllb[:60], "verif_out": verif[:60],
                })

    if cite_total > 0:
        nllb_pct  = nllb_cite_ok  / cite_total * 100
        verif_pct = verif_cite_ok / cite_total * 100
        print(f"   NLLB alone:            {nllb_cite_ok}/{cite_total} ({nllb_pct:.1f}%)")
        print(f"   NLLB + Llama verified: {verif_cite_ok}/{cite_total} ({verif_pct:.1f}%)")
        print(f"   Improvement:           {verif_pct - nllb_pct:+.1f}%")
        if cite_results:
            print(f"\n   Failed citations:")
            for r in cite_results[:5]:
                print(f"\n   Citation: '{r['citation']}'")
                print(f"   NLLB:     {'✅' if r['nllb_ok'] else '❌'} '{r['nllb_out']}'")
                print(f"   Verified: {'✅' if r['verif_ok'] else '❌'} '{r['verif_out']}'")
    else:
        print("   No legal citations found in test spans")

    # ── 3. Critical error detection ───────────────────────────
    print(f"\n\n🧠 SECTION 3: Critical Error Detection")
    print("-" * 65)
    print("   (Cases where NLLB produced clearly wrong output)\n")

    KNOWN_HALLUCINATIONS = {
        "docket no":  ["no, no lo", "no, it"],
        "docket no.": ["no, no lo", "no, it"],
    }
    critical_errors = []

    for orig, nllb, verif in zip(original_texts, nllb_translations,
                                  verified_translations):
        nllb_ratio = len(nllb) / max(len(orig), 1)
        if nllb_ratio < 0.2 or nllb_ratio > 3.0:
            critical_errors.append({
                "type": "length_anomaly", "orig": orig[:50],
                "nllb": nllb[:50], "verif": verif[:50],
                "ratio": nllb_ratio,
            })
        for en_term in ["defendant", "plaintiff", "arraignment",
                        "docket", "waiver", "bail"]:
            if (en_term in orig.lower() and en_term in nllb.lower()
                    and en_term not in verif.lower()):
                critical_errors.append({
                    "type": "untranslated_term", "orig": orig[:50],
                    "nllb": nllb[:50], "verif": verif[:50],
                    "term": en_term,
                })
        for trigger, bad_patterns in KNOWN_HALLUCINATIONS.items():
            if trigger in orig.lower():
                for bad in bad_patterns:
                    if bad in nllb.lower():
                        critical_errors.append({
                            "type": "known_hallucination", "orig": orig[:50],
                            "nllb": nllb[:50], "verif": verif[:50],
                            "term": trigger,
                        })

    if critical_errors:
        print(f"   ⚠️  {len(critical_errors)} critical errors caught by Llama:")
        for e in critical_errors[:5]:
            if e["type"] == "length_anomaly":
                print(f"\n   Type: Hallucination (ratio={e['ratio']:.1f})")
            elif e["type"] == "known_hallucination":
                print(f"\n   Type: Known hallucination (trigger='{e['term']}')")
            else:
                print(f"\n   Type: Untranslated term ('{e['term']}')")
            print(f"   Original: '{e['orig']}'")
            print(f"   NLLB:     '{e['nllb']}'")
            print(f"   Fixed:    '{e['verif']}'")
    else:
        print("   ✅ No critical errors detected")

    # ── 4. Summary ────────────────────────────────────────────
    print(f"\n\n{'=' * 65}")
    print("📋 SUMMARY")
    print('=' * 65)

    if total_terms > 0 and cite_total > 0:
        nllb_overall  = (nllb_correct + nllb_cite_ok) / (total_terms + cite_total) * 100
        verif_overall = (verif_correct + verif_cite_ok) / (total_terms + cite_total) * 100
        print(f"   Combined term+citation accuracy:")
        print(f"   NLLB alone:            {nllb_overall:.1f}%")
        print(f"   NLLB + Llama verified: {verif_overall:.1f}%")
        print(f"   Improvement:           {verif_overall - nllb_overall:+.1f}%")
    elif total_terms > 0:
        print(f"   Term accuracy (no citations found):")
        print(f"   NLLB alone:            {nllb_correct/total_terms*100:.1f}%")
        print(f"   NLLB + Llama verified: {verif_correct/total_terms*100:.1f}%")
    elif cite_total > 0:
        print(f"   Citation accuracy (no glossary terms found):")
        print(f"   NLLB alone:            {nllb_cite_ok/cite_total*100:.1f}%")
        print(f"   NLLB + Llama verified: {verif_cite_ok/cite_total*100:.1f}%")

    print(f"   Critical errors caught: {len(critical_errors)}")

    try:
        print(f"   BERTScore (semantic):  "
              f"{nllb_mean*100:.1f}% → {verified_mean*100:.1f}% "
              f"({(verified_mean-nllb_mean)*100:+.2f}%)")
    except NameError:
        print("   BERTScore: run RTT test cell first to see live values")

    print(f"\n   NOTE: BERTScore = semantic similarity.")
    print(f"   Term accuracy = legal correctness.")
    print(f"   For court documents, term accuracy is what matters.")


print("🧪 Running term-level accuracy test...")
run_term_level_accuracy(original_texts, nllb_spanish, verified_spanish)

🧪 Running term-level accuracy test...


NameError: name 'original_texts' is not defined

# Cell 8 - Presidio PII redaction

This cell defines the Presidio PII redaction system. It is not currently in use, since blank legal forms don't contain PII, but can be used to redact PII from the translated text in the future.

In [ ]:
# ============================================================
# CELL 8: PRESIDIO PII REDACTION
# Detects and redacts PII in text spans and PDF form widgets.
# Legal citations and section references are always preserved.
# Set REDACT_PII = False for personal-use translation.
# ============================================================
from presidio_analyzer import AnalyzerEngine
from presidio_anonymizer import AnonymizerEngine
from presidio_anonymizer.entities import OperatorConfig

print("⏳ Loading Presidio...")
analyzer   = AnalyzerEngine()
anonymizer = AnonymizerEngine()
print("✅ Presidio ready")

REDACT_PII   = True   # set False for personal use
REDACT_DATES = False  # keep dates visible in output

PII_ENTITIES = [
    "PERSON", "PHONE_NUMBER", "EMAIL_ADDRESS", "US_SSN",
    "CREDIT_CARD", "US_PASSPORT", "US_DRIVER_LICENSE",
    "IP_ADDRESS", "DATE_TIME",
]
if not REDACT_DATES and "DATE_TIME" in PII_ENTITIES:
    PII_ENTITIES.remove("DATE_TIME")

PRESERVE_PATTERNS = [
    r"G\.L\.\s+c\.\s+\d+",    r"§\s*\d+[A-Z]?",
    r"\[\d+[\.\(\)a-zA-Z\;]+\]", r"Mass\.\s+R\.\s+Crim\.",
    r"Paragraph\s+\d+",         r"TC\d+",
    r"P\.\s*\d+\.\d+",
]

def _should_preserve(text: str) -> bool:
    return any(re.search(p, text, re.IGNORECASE) for p in PRESERVE_PATTERNS)

def redact_pii(text: str) -> tuple:
    if not REDACT_PII or not text.strip() or _should_preserve(text):
        return text, []
    try:
        results = analyzer.analyze(text=text, entities=PII_ENTITIES, language="en")
    except Exception:
        return text, []
    if not results:
        return text, []
    found = list(set(r.entity_type for r in results))
    anon  = anonymizer.anonymize(
        text=text, analyzer_results=results,
        operators={e: OperatorConfig("replace", {"new_value": f"[{e}]"}) for e in found}
    )
    return anon.text, found

def redact_spans(original_texts: list, translated_texts: list) -> list:
    if not REDACT_PII:
        return translated_texts
    redacted = list(translated_texts)
    total    = 0
    for i, (orig, trans) in enumerate(zip(original_texts, translated_texts)):
        _, entities = redact_pii(orig)
        if not entities:
            continue
        redacted[i], _ = redact_pii(trans)
        total += 1
    if total == 0:
        print(f"   ✅ No PII found in {len(original_texts)} spans")
    else:
        print(f"   🔒 Redacted PII in {total}/{len(original_texts)} spans")
    return redacted

def redact_form_fields(page) -> int:
    if not REDACT_PII:
        return 0
    count = 0
    for w in list(page.widgets()):
        val = str(w.field_value or "").strip()
        if not val:
            continue
        redacted, entities = redact_pii(val)
        if entities:
            w.field_value = redacted
            w.update()
            if w.rect:
                page.add_redact_annot(w.rect, fill=(1, 1, 1))
            count += 1
    if count > 0:
        page.apply_redactions(images=pymupdf.PDF_REDACT_IMAGE_NONE)
    return count

print("✅ PII redaction ready")



# Cell 9 - Layout preserving translation

In [ ]:
# ============================================================
# CELL 9 — LAYOUT-PRESERVING TRANSLATION v23
#
# FIXES OVER v22:
#   FIX-1 : _LIST_START_RE       — space after bullet optional
#   FIX-2 : _presplit_spans      — mid-span □/•/◦ pre-split
#   FIX-3 : _clip_redact_rect    — 0.5 pt inward shrink
#   FIX-4 : _scaled_font_size    — heading height cap + 5pt floor
#   FIX-5 : column splitting     — dynamic gap + narrow-cell guard
#   FIX-6 : checkbox mid-span    — full sub-span generation
#   FIX-7 : downward expansion   — y-ceiling from next unit
#   FIX-8 : container-width rect — full horizontal budget
#   FIX-9 : hard cell boundary   — table-cell text never escapes
#            drawn container; in_table_cell flag on every unit;
#            reflow y-push propagated column-wise for free text
#   FIX-10: checkbox threshold   — 20 pt + square-shape guard
# ============================================================

import pymupdf
import re
import os

NLLB_CHAR_LIMIT = 380
MIN_CHUNK_SIZE  = 120

# ── List marker patterns ──────────────────────────────────────
_BULLET_CHARS = set('•◦○▪▫□■▸▹–—☐☑✓✔')
_LIST_START_RE = re.compile(
    r'^\s*(?:'
    r'[•◦○▪▫□■▸▹–—☐☑✓✔]\s*'
    r'|\(?[0-9]+[.)]\s'
    r'|\(?[a-z][.)]\s'
    r'|[o]\s{2,}'
    r')',
    re.UNICODE
)

_UNICODE_CHECKBOX_CHARS = set('□☐☑✓✔■▪•◦·▸▹')


def _color_hex(color):
    if isinstance(color, tuple) and len(color) == 3:
        r, g, b = (max(0, min(255, int(c * 255))) for c in color)
        return f"#{r:02x}{g:02x}{b:02x}"
    return "#000000"


def _detect_alignment(spans, cell_rect):
    if not spans or cell_rect.width <= 0: return 'left'
    lm = [s["bbox"][0] - cell_rect.x0 for s in spans]
    rm = [cell_rect.x1 - s["bbox"][2] for s in spans]
    al, ar = sum(lm)/len(lm), sum(rm)/len(rm)
    if al > 3 and ar > 3 and abs(al-ar) < 20: return 'center'
    if ar < 8 and al > 20: return 'right'
    return 'left'


def _spans_to_style_groups(spans: list) -> list:
    groups = []
    for span in spans:
        t = span.get("text", "")
        if not t: continue
        family, bold, italic = get_css_font(span)
        size  = get_font_size(span)
        color = _safe_color(span)
        if (groups
                and groups[-1]["family"] == family
                and groups[-1]["bold"]   == bold
                and groups[-1]["italic"] == italic
                and abs(groups[-1]["size"] - size) < 0.5
                and groups[-1]["color"]  == color):
            groups[-1]["text"] += t
        else:
            groups.append({"text": t, "family": family,
                           "bold": bold, "italic": italic,
                           "size": size, "color": color})
    return groups


def _dominant_size(sg):
    if not sg: return 10.0
    return max(sg, key=lambda s: len(s["text"]))["size"]

def _dominant_family(sg):
    if not sg: return "Helvetica, Arial, sans-serif"
    return max(sg, key=lambda s: len(s["text"]))["family"]

def _fc_from_family(fam):
    if "monospace" in fam: return "cour"
    if "serif" in fam: return "tiro"
    return "helv"


def _scaled_font_size(text, sgs, cw, ch, min_size=5.0):
    if not sgs or cw <= 4: return _dominant_size(sgs)
    orig = _dominant_size(sgs)
    fc   = _fc_from_family(_dominant_family(sgs))
    if ch > 0:
        orig = min(orig, ch * 0.9)
    effective_min = max(min_size, min(orig * 0.55, 5.0))

    def _lines(sz):
        try:
            f = pymupdf.Font(fc); words = text.split()
            lines, cur = 1, 0.0; sp = f.text_length(" ", fontsize=sz)
            for w in words:
                ww = f.text_length(w, fontsize=sz)
                if cur > 0 and cur+sp+ww > cw*0.93: lines += 1; cur = ww
                else: cur += (sp if cur > 0 else 0) + ww
            return lines
        except Exception:
            cpl = max(1, int(cw/max(sz*0.55, 1)))
            return max(1, -(-len(text)//cpl))

    sz = orig
    while sz >= effective_min:
        if _lines(sz)*sz*1.15 <= ch*1.1: return sz
        sz -= 0.5
    return effective_min


def _estimate_text_bottom(text, sgs, rect, scaled):
    fc = _fc_from_family(_dominant_family(sgs))
    try:
        f = pymupdf.Font(fc); words = text.split()
        lines, cur = 1, 0.0; sp = f.text_length(" ", fontsize=scaled)
        for w in words:
            ww = f.text_length(w, fontsize=scaled)
            if cur > 0 and cur+sp+ww > rect.width*0.93: lines += 1; cur = ww
            else: cur += (sp if cur > 0 else 0) + ww
        return rect.y0 + lines*scaled*1.15
    except Exception:
        return rect.y1


def _build_rich_html(sgs, text, centered, right,
                     fs_override=None, top_pad=0.0):
    if not sgs or not text.strip(): return ""
    align = "center" if centered else ("right" if right else "left")
    ps = (f"font-family:{sgs[0]['family']};text-align:{align};"
          f"margin:0;padding:0;padding-top:{top_pad:.1f}pt;line-height:1.15;")
    if len(sgs) == 1:
        mapped = [{**sgs[0], "text": text}]
    else:
        ol = [max(len(s["text"].strip()),1) for s in sgs]; ot = sum(ol)
        tl = len(text)
        splits = []; cumsum = 0
        for length in ol[:-1]:
            cumsum += length
            tp = int(cumsum/ot*tl)
            while tp < tl and text[tp] != ' ': tp += 1
            if splits and tp <= splits[-1]: tp = splits[-1]+1
            splits.append(min(tp, tl))
        mapped = []; prev = 0
        for i, sp in enumerate(splits):
            seg = text[prev:sp].strip()
            if seg: mapped.append({**sgs[i], "text": seg})
            prev = sp
        rem = text[prev:].strip()
        if rem and len(sgs) > len(splits):
            mapped.append({**sgs[len(splits)], "text": rem})
        elif rem and mapped:
            mapped[-1]["text"] += " " + rem

    parts = []
    for sg in mapped:
        safe = sg["text"]
        safe = safe.replace('\u201c', '"').replace('\u201d', '"')
        safe = safe.replace('\u2018', "'").replace('\u2019', "'")
        safe = safe.replace('\u00ab', '"').replace('\u00bb', '"')
        safe = safe.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
        if not safe: continue
        orig_letters = [c for c in sg.get("_orig_text", sg["text"]) if c.isalpha()]
        if orig_letters and all(c.isupper() for c in orig_letters):
            safe = safe.upper()
        sz = fs_override if fs_override else sg["size"]
        ss = (f"font-family:{sg['family']};font-size:{sz:.1f}pt;"
              f"font-weight:{'bold' if sg['bold'] else 'normal'};"
              f"font-style:{'italic' if sg['italic'] else 'normal'};"
              f"color:{_color_hex(sg['color'])};")
        parts.append(f'<span style="{ss}">{safe}</span>')
    if not parts: return ""
    return f'<p style="{ps}">{" ".join(parts)}</p>'


# ════════════════════════════════════════════════════════════
# FIX-9/10: COMBINED INSERT WITH HARD BOUNDARY + REFLOW
# ════════════════════════════════════════════════════════════

def _insert_cell_unit(page, unit, max_y1=None, y_push=0.0):
    """
    FIX-9 + FIX-10: Two-mode insertion.

    HARD BOUNDARY MODE  (unit["in_table_cell"] is True)
      — avail rect is strictly clipped to the drawn container walls.
      — font shrinks iteratively until text fits inside cell height.
      — text NEVER escapes the drawn cell boundary.
      — returns (actual_bottom, expansion=0).

    REFLOW MODE  (unit["in_table_cell"] is False)
      — y_push offset is applied so this unit slides down past any
        prior unit that expanded in the same column.
      — rect expands downward up to max_y1 ceiling if needed.
      — returns (actual_bottom, expansion_added).
    """
    cr  = pymupdf.Rect(unit["cell_rect"])
    co  = pymupdf.Rect(unit.get("container_rect", cr))
    tt  = unit.get("trans_text", "")
    sgs = unit.get("style_groups", [])
    in_table = unit.get("in_table_cell", False)

    if not tt or not tt.strip() or not sgs:
        return cr.y1 + y_push, 0.0
    if cr.width <= 2 or cr.height <= 2:
        return cr.y1 + y_push, 0.0

    # Apply accumulated y-push from prior reflow units in same column
    cr = pymupdf.Rect(cr.x0, cr.y0 + y_push, cr.x1, cr.y1 + y_push)
    co = pymupdf.Rect(co.x0, co.y0 + y_push, co.x1, co.y1 + y_push)

    if in_table:
        # ── Hard boundary: clip to drawn container ──────────────
        # FIX-11a: use co.y1 (container bottom) NOT cr.y1 (span bottom).
        # cr.y1 is only where the English text ended inside the cell —
        # using it as the height limit starves Spanish text of vertical
        # space, forces the font to shrink to minimum, and leaves a large
        # blank gap between the text and the cell's actual bottom wall.
        avail = pymupdf.Rect(
            max(cr.x0, co.x0 + 0.5),   # left:  max of span-left and cell-left
            cr.y0,                       # top:   where this text block starts
            min(co.x1 - 0.5, cr.x1 + (co.x1 - cr.x1)),  # right: cell right wall
            co.y1 - 0.5,                # bottom: full container bottom
        )
        # Clamp right edge to actual container
        avail = pymupdf.Rect(
            max(cr.x0, co.x0 + 0.5),
            cr.y0,
            co.x1 - 0.5,
            co.y1 - 0.5,
        )
        if avail.width < 4 or avail.height < 4:
            return cr.y1, 0.0

        # Scale using actual available width and the remaining cell height
        sc = _scaled_font_size(tt, sgs, avail.width, avail.height)
        # Iteratively shrink until text fits within cell bottom
        for _ in range(40):
            if _estimate_text_bottom(tt, sgs, avail, sc) <= avail.y1 + 0.5:
                break
            sc = max(sc - 0.4, 4.0)
            if sc <= 4.0: break

        insert_rect = avail
        expansion   = 0.0

    else:
        # ── Reflow mode: expand downward if needed ───────────────
        avail = pymupdf.Rect(cr.x0, cr.y0, co.x1, cr.y1)
        if avail.width < 4:
            avail = pymupdf.Rect(cr.x0, cr.y0, cr.x1, cr.y1)

        sc = _scaled_font_size(tt, sgs, avail.width,
                               max(co.height, cr.height))
        est = _estimate_text_bottom(tt, sgs, avail, sc)

        if est > avail.y1 + 1.0:
            ceiling   = (max_y1 - 0.5) if max_y1 is not None \
                        else (page.rect.y1 - 1.0)
            new_y1    = min(est + 2.0, ceiling)
            expansion = max(0.0, new_y1 - cr.y1)
            avail     = pymupdf.Rect(avail.x0, avail.y0,
                                     avail.x1, new_y1)
        else:
            expansion = 0.0

        insert_rect = avail

    tagged = [{**sg, "_orig_text": sg["text"]} for sg in sgs]
    html = _build_rich_html(
        tagged, tt,
        unit.get("in_centered", False),
        unit.get("is_right",   False),
        fs_override=sc,
        top_pad=unit.get("top_pad", 0.0)
    )
    try:
        page.insert_htmlbox(insert_rect, html)
    except Exception:
        try:
            sg = sgs[0]; fc = _fc_from_family(sg.get("family", ""))
            if sg["bold"] and sg["italic"]:
                fc = {"cour":"cobi","tiro":"tibi","helv":"hebi"}.get(fc, fc)
            elif sg["bold"]:
                fc = {"cour":"cobo","tiro":"tibo","helv":"hebo"}.get(fc, fc)
            elif sg["italic"]:
                fc = {"cour":"coit","tiro":"tiit","helv":"heit"}.get(fc, fc)
            page.insert_text(
                pymupdf.Point(insert_rect.x0, insert_rect.y0 + sc),
                tt, fontsize=sc, fontname=fc, color=sg["color"])
        except Exception: pass

    actual_bottom = _estimate_text_bottom(tt, sgs, insert_rect, sc)
    return actual_bottom, expansion


def _union_rects(rects):
    if not rects: return pymupdf.Rect()
    return pymupdf.Rect(min(r.x0 for r in rects), min(r.y0 for r in rects),
                        max(r.x1 for r in rects), max(r.y1 for r in rects))


# ════════════════════════════════════════════════════════════
# DRAWING-AWARE STRUCTURAL LAYER
# ════════════════════════════════════════════════════════════

def _classify_drawings(page):
    pa = page.rect.width * page.rect.height
    res = {"page_border":[],"checkboxes":[],"underlines":[],
           "table_cells":[], "all_protected":[]}
    for d in page.get_drawings():
        r = d.get("rect")
        if r is None: continue
        r = pymupdf.Rect(r); w, h = r.width, r.height
        if w < 1 or h < 1: continue
        info = {"rect":r, "color":d.get("color") or (0,0,0),
                "fill":d.get("fill"), "width":d.get("width", 0.5)}
        cov = (w*h)/pa if pa > 0 else 0
        if   cov > 0.55:
            res["page_border"].append(r)
        # FIX-10: enlarged to 20pt + square-shape guard (|w-h|<8)
        # prevents thin table rules from being mis-classified
        elif w <= 20 and h <= 20 and abs(w - h) < 8:
            res["checkboxes"].append(info)
        elif h < 3 and w > 20:
            res["underlines"].append(info)
        else:
            res["table_cells"].append(r)
        res["all_protected"].append(r)

    res["table_cells"].sort(key=lambda r: r.width*r.height)
    return res


def _rects_equal(r1, r2, tol=1.0):
    if r1 is None or r2 is None: return r1 is r2
    return (abs(r1.x0-r2.x0)<tol and abs(r1.y0-r2.y0)<tol
            and abs(r1.x1-r2.x1)<tol and abs(r1.y1-r2.y1)<tol)

def _find_tightest_table_cell(sr, cells, pad=2.0):
    best, ba = None, float("inf")
    for c in cells:
        e = pymupdf.Rect(c.x0-pad, c.y0-pad, c.x1+pad, c.y1+pad)
        if e.contains(sr):
            a = c.width*c.height
            if a < ba: ba = a; best = c
    return best

def _find_underlines_for_span(sb, underlines, vtol=12.0):
    hits = []
    for ul in underlines:
        r = ul["rect"]
        if not (sb[3] <= r.y0 <= sb[3]+vtol): continue
        if r.x1 < sb[0] or r.x0 > sb[2]: continue
        hits.append(ul)
    return hits

def _find_associated_checkbox(tr, cbs, max_gap=25.0):
    cy = (tr.y0+tr.y1)/2; best, bg = None, max_gap+1
    for cb in cbs:
        r = cb["rect"]; hg = tr.x0-r.x1
        if r.x1 > tr.x0 or hg > max_gap: continue
        if abs((r.y0+r.y1)/2-cy) > 8.0: continue
        if hg < bg: bg = hg; best = cb
    return (best, tr.x0-best["rect"].x1) if best else (None, 0.0)

def _find_inline_checkboxes(spans, cbs, vtol=8.0):
    if not spans or not cbs: return []
    hits = []
    for cb in cbs:
        r = cb["rect"]; cx = (r.x0+r.x1)/2; ccy = (r.y0+r.y1)/2
        sy0 = min(s["bbox"][1] for s in spans)
        sy1 = max(s["bbox"][3] for s in spans)
        if abs(ccy-(sy0+sy1)/2) > vtol: continue
        sx0 = min(s["bbox"][0] for s in spans)
        sx1 = max(s["bbox"][2] for s in spans)
        if cx < sx0-20 or cx > sx1+20: continue
        bi = 0
        for i, s in enumerate(spans):
            if cx < s["bbox"][0]: bi = i; break
            bi = i+1
        hits.append((bi, cb))
    hits.sort(key=lambda h: h[0]); return hits

def _get_fill_lines_for_cluster(cr, co, underlines):
    hits = []; cy = (cr.y0+cr.y1)/2; ch = max(cr.height, 4.0)
    cx1 = co.x1 if co else cr.x1+200
    cy1 = co.y1 if co else cr.y1+50
    for ul in underlines:
        r = ul["rect"]; uc = (r.y0+r.y1)/2
        if (r.x0 >= cr.x1-5 and abs(uc-cy) <= ch*0.9 and r.x0 <= cx1+4):
            hits.append(ul); continue
        if (r.y0 >= cr.y1-2 and r.y0 <= cy1+4
                and r.x0 <= cx1+4 and r.x1 >= cr.x0-10):
            hits.append(ul)
    return hits


# ── Fill-field detection ──────────────────────────────────────
_FILL_FIELD_RE = re.compile(r'_{3,}')

def _is_fill_field_span(span):
    t = span.get("text", "").strip()
    if not t: return False
    underscores = sum(1 for c in t if c == '_')
    return underscores >= 3 and underscores / max(len(t), 1) > 0.5

def _split_at_fill_fields(spans):
    if not spans: return []
    segments = []; current = []; current_type = None
    for span in spans:
        text = span.get("text", "")
        if not text: continue
        underscore_runs = list(re.finditer(r'_{3,}', text))
        if not underscore_runs:
            span_type = "text"
            if span_type != current_type and current:
                segments.append({"type": current_type, "spans": current})
                current = []
            current.append(span); current_type = span_type
        elif _is_fill_field_span(span) and not any(c.isalpha() for c in text):
            span_type = "fill"
            if span_type != current_type and current:
                segments.append({"type": current_type, "spans": current})
                current = []
            current.append(span); current_type = span_type
        else:
            bbox = span["bbox"]
            span_width = bbox[2] - bbox[0]
            text_len = max(len(text), 1)
            char_width = span_width / text_len
            pos = 0
            for run in underscore_runs:
                if run.start() > pos:
                    text_part = text[pos:run.start()]
                    if text_part.strip():
                        sub_x0 = bbox[0] + pos * char_width
                        sub_x1 = bbox[0] + run.start() * char_width
                        sub_span = {**span, "text": text_part,
                                    "bbox": (sub_x0, bbox[1], sub_x1, bbox[3])}
                        if current_type != "text" and current:
                            segments.append({"type": current_type, "spans": current})
                            current = []
                        current.append(sub_span); current_type = "text"
                fill_x0 = bbox[0] + run.start() * char_width
                fill_x1 = bbox[0] + run.end() * char_width
                fill_span = {**span, "text": run.group(),
                             "bbox": (fill_x0, bbox[1], fill_x1, bbox[3])}
                if current_type != "fill" and current:
                    segments.append({"type": current_type, "spans": current})
                    current = []
                current.append(fill_span); current_type = "fill"
                pos = run.end()
            if pos < len(text):
                text_part = text[pos:]
                if text_part.strip():
                    sub_x0 = bbox[0] + pos * char_width
                    sub_span = {**span, "text": text_part,
                                "bbox": (sub_x0, bbox[1], bbox[2], bbox[3])}
                    if current_type != "text" and current:
                        segments.append({"type": current_type, "spans": current})
                        current = []
                    current.append(sub_span); current_type = "text"
    if current:
        segments.append({"type": current_type, "spans": current})
    return segments


# ════════════════════════════════════════════════════════════
# FIX-2: PRE-SPLIT SPANS AT INLINE CHECKBOX/BULLET CHARS
# ════════════════════════════════════════════════════════════

def _presplit_spans_at_checkboxes(spans):
    result = []
    for span in spans:
        text = span.get("text", "")
        if not text:
            continue
        split_positions = []
        for i, c in enumerate(text):
            if c in _UNICODE_CHECKBOX_CHARS and i > 0:
                prev = text[i-1]
                if prev in (' ', '\t', '\u00a0') or prev in _UNICODE_CHECKBOX_CHARS:
                    split_positions.append(i)
        if not split_positions:
            result.append(span); continue
        bbox = span["bbox"]
        span_width = max(bbox[2] - bbox[0], 1.0)
        char_width = span_width / max(len(text), 1)
        prev = 0
        for pos in split_positions:
            seg_text = text[prev:pos]
            if seg_text:
                sub_x0 = bbox[0] + prev * char_width
                sub_x1 = bbox[0] + pos  * char_width
                result.append({**span, "text": seg_text,
                               "bbox": (sub_x0, bbox[1], sub_x1, bbox[3])})
            prev = pos
        if prev < len(text):
            sub_x0 = bbox[0] + prev * char_width
            result.append({**span, "text": text[prev:],
                           "bbox": (sub_x0, bbox[1], bbox[2], bbox[3])})
    return result


# ════════════════════════════════════════════════════════════
# RC4 + FIX-6: UNICODE CHECKBOX DETECTION
# ════════════════════════════════════════════════════════════

def _has_unicode_checkbox(text):
    return any(c in _UNICODE_CHECKBOX_CHARS for c in text)

def _find_unicode_checkbox_splits(spans):
    result = []
    for span in spans:
        text = span.get("text", "").strip()
        if not text:
            result.append(span); continue
        mid_positions = [j for j, c in enumerate(text)
                         if c in _UNICODE_CHECKBOX_CHARS and j > 0]
        if not mid_positions:
            result.append(span); continue
        bbox = span["bbox"]
        span_width = max(bbox[2] - bbox[0], 1.0)
        char_width = span_width / max(len(text), 1)
        prev = 0
        for pos in mid_positions:
            seg = text[prev:pos]
            if seg:
                sx0 = bbox[0] + prev * char_width
                sx1 = bbox[0] + pos  * char_width
                result.append({**span, "text": seg,
                               "bbox": (sx0, bbox[1], sx1, bbox[3])})
            prev = pos
        if prev < len(text):
            sx0 = bbox[0] + prev * char_width
            result.append({**span, "text": text[prev:],
                           "bbox": (sx0, bbox[1], bbox[2], bbox[3])})
    return result


# ════════════════════════════════════════════════════════════
# RC3: LIST STRUCTURE DETECTION
# ════════════════════════════════════════════════════════════

def _get_line_text(line):
    return " ".join(s["text"] for s in line["spans"]).strip()

def _is_list_line(text):
    return bool(_LIST_START_RE.match(text))

def _detect_list_structure(lines):
    if len(lines) < 2: return "none"
    bullet_count = semi_count = sub_bullet_count = 0
    for ln in lines:
        text = _get_line_text(ln)
        if not text: continue
        if _is_list_line(text): bullet_count += 1
        if text.rstrip().endswith(';') or text.rstrip().endswith('; and'):
            semi_count += 1
        stripped = text.lstrip()
        if (stripped.startswith('o ') or stripped.startswith('- ')) \
                and len(text) > len(stripped) + 3:
            sub_bullet_count += 1
    if bullet_count    >= 2: return "bullet_list"
    if sub_bullet_count >= 2: return "sub_bullet"
    if semi_count      >= 2: return "semicolon_list"
    return "none"


# ════════════════════════════════════════════════════════════
# RC2: PARAGRAPH SPLITTING (multi-signal)
# ════════════════════════════════════════════════════════════

def _line_is_bold(line):
    bold_chars = total_chars = 0
    for s in line["spans"]:
        t = s.get("text", "").strip()
        if not t: continue
        n = len(t); total_chars += n
        flags = s.get("flags", 0)
        fn = s.get("font", "").lower()
        if (flags & 16) or any(w in fn for w in ["bold","black","heavy"]):
            bold_chars += n
    return bold_chars > total_chars * 0.5 if total_chars > 0 else False

def _line_is_centered(line, container_width, container_x0=0):
    spans = [s for s in line["spans"] if s["text"].strip()]
    if not spans: return False
    lx = min(s["bbox"][0] for s in spans) - container_x0
    rx = container_width - (max(s["bbox"][2] for s in spans) - container_x0)
    return lx > 15 and rx > 15 and abs(lx - rx) < 30

def _line_font_size(line):
    sizes = []
    for s in line["spans"]:
        t = s.get("text", "").strip()
        if t: sizes.extend([get_font_size(s)] * len(t))
    return sum(sizes)/len(sizes) if sizes else 10.0


def _group_lines_into_paragraphs(lines, container=None):
    if not lines: return []
    list_type = _detect_list_structure(lines)
    if list_type in ("bullet_list", "semicolon_list", "sub_bullet"):
        return [[ln] for ln in lines]

    cw  = container.width  if container else 500
    cx0 = container.x0    if container else 0

    paras = [[lines[0]]]
    for i in range(1, len(lines)):
        prev_ln = lines[i-1]; curr_ln = lines[i]
        prev_spans = [s for s in prev_ln["spans"] if s["text"].strip()]
        curr_spans = [s for s in curr_ln["spans"] if s["text"].strip()]
        if not prev_spans or not curr_spans:
            paras[-1].append(curr_ln); continue

        prev_y1 = max(s["bbox"][3] for s in prev_spans)
        curr_y0 = min(s["bbox"][1] for s in curr_spans)
        gap = curr_y0 - prev_y1

        prev_size = _line_font_size(prev_ln)
        curr_size = _line_font_size(curr_ln)
        local_size = min(prev_size, curr_size)
        gap_threshold = local_size * 2.0

        prev_bold = _line_is_bold(prev_ln); curr_bold = _line_is_bold(curr_ln)
        prev_cen  = _line_is_centered(prev_ln, cw, cx0)
        curr_cen  = _line_is_centered(curr_ln, cw, cx0)

        large_gap   = gap > gap_threshold
        style_break = ((prev_bold and not curr_bold) or
                       (prev_cen  and not curr_cen)  or
                       (not prev_bold and curr_bold)  or
                       (not prev_cen  and curr_cen))
        size_change = abs(prev_size - curr_size) > 2.0

        should_split = False
        if large_gap and style_break:
            should_split = True
        elif style_break and not large_gap:
            prev_text = _get_line_text(prev_ln)
            curr_text = _get_line_text(curr_ln)
            if (prev_bold or prev_cen) and len(prev_text) < 80:
                should_split = True
            if (curr_bold or curr_cen) and len(curr_text) < 80:
                should_split = True
        elif large_gap and not style_break:
            prev_group_lines = len(paras[-1])
            curr_text = _get_line_text(curr_ln)
            if   prev_group_lines >= 4: should_split = True
            elif len(curr_text) > 60 and prev_group_lines >= 2: should_split = True
            elif local_size < 8.0: should_split = False
        elif size_change:
            should_split = True

        # FIX-8d: same-style continuation guard
        if should_split:
            same_style = (
                prev_bold == curr_bold
                and prev_cen == curr_cen
                and abs(prev_size - curr_size) < 1.0
                and gap < local_size * 1.8
                and (prev_bold or prev_cen)
            )
            if same_style: should_split = False

        paras.append([curr_ln]) if should_split else paras[-1].append(curr_ln)

    return paras


def _split_long_text(text, sgs, max_chars=NLLB_CHAR_LIMIT,
                     min_chars=MIN_CHUNK_SIZE):
    if len(text) <= max_chars: return [text]
    boundaries = []
    for m in re.finditer(r'(?<=[.!?])\s+(?=[A-Z""\(])', text):
        boundaries.append(m.end())
    if not boundaries:
        for m in re.finditer(r'(?<=[.!?])\s+', text): boundaries.append(m.end())
    if not boundaries:
        for m in re.finditer(r'(?<=[,;:])\s+', text): boundaries.append(m.end())
    if not boundaries: return [text]
    chunks = []; start = 0
    for bpos in boundaries:
        chunk = text[start:bpos].strip(); remaining = text[bpos:].strip()
        if len(chunk) >= max_chars:
            chunks.append(chunk); start = bpos
        elif len(chunk) >= min_chars and len(remaining) >= min_chars:
            chunks.append(chunk); start = bpos
    remainder = text[start:].strip()
    if remainder:
        if chunks and len(remainder) < min_chars:
            chunks[-1] = chunks[-1] + " " + remainder
        else: chunks.append(remainder)
    return chunks if chunks else [text]


# ── Intra-block splitting (FIX-5: dynamic gap, narrow-cell guard) ─
def _split_spans_into_columns(blk_spans, cell_rect, blk_rect=None,
                               table_cells=None, checkboxes=None,
                               gap_threshold=None, size_threshold=2.0):
    if not blk_spans: return []
    cells = table_cells or []; cbs = checkboxes or []

    if gap_threshold is None:
        cw = cell_rect.width if cell_rect else 500.0
        gap_threshold = max(6.0, min(18.0, cw * 0.03))

    narrow_cell = cell_rect is not None and cell_rect.width < 80.0

    def _span_cell(s):
        return _find_tightest_table_cell(pymupdf.Rect(s["bbox"]), cells, 2.0)

    def _checkbox_between(s1, s2):
        s1_x1 = s1["bbox"][2]; s2_x0 = s2["bbox"][0]
        s1_cy = (s1["bbox"][1]+s1["bbox"][3])/2
        for cb in cbs:
            r = cb["rect"]; cb_cx = (r.x0+r.x1)/2; cb_cy = (r.y0+r.y1)/2
            if s1_x1-5 <= cb_cx <= s2_x0+5 and abs(cb_cy-s1_cy) < 12:
                return True
        return False

    def _unicode_cb_between(s1, s2):
        t2 = s2.get("text","").lstrip()
        return len(t2) > 0 and t2[0] in _UNICODE_CHECKBOX_CHARS

    line_bands = []
    for span in blk_spans:
        sb = span["bbox"]; placed = False
        for band in line_bands:
            by0 = min(s["bbox"][1] for s in band)
            by1 = max(s["bbox"][3] for s in band)
            if sb[1] <= by1+4 and sb[3] >= by0-4:
                band.append(span); placed = True; break
        if not placed: line_bands.append([span])

    def _band_size(b):
        return max(get_font_size(s) for s in b) if b else 10.0

    vert_groups = []
    if line_bands:
        cg = [line_bands[0]]; cs = _band_size(line_bands[0])
        for band in line_bands[1:]:
            bs = _band_size(band)
            py1 = max(s["bbox"][3] for s in cg[-1])
            by0 = min(s["bbox"][1] for s in band)
            if abs(bs-cs) > size_threshold and by0 >= py1-2:
                vert_groups.append(list(cg)); cg = [band]; cs = bs
            else: cg.append(band)
        vert_groups.append(cg)

    if blk_rect is not None and cell_rect is not None:
        rb = min(cell_rect.x1, blk_rect.x1+2)
    elif cell_rect is not None: rb = cell_rect.x1
    else: rb = None

    all_cl = []
    for vg in vert_groups:
        bcr = []
        for band in vg:
            bx = sorted(band, key=lambda s: s["bbox"][0])
            if narrow_cell:
                bcr.append([bx]); continue
            hc = [[bx[0]]]; pc = _span_cell(bx[0])
            for s in bx[1:]:
                cc  = _span_cell(s)
                dc  = (pc is not None and cc is not None and not _rects_equal(pc, cc))
                g   = s["bbox"][0] - hc[-1][-1]["bbox"][2]
                gs  = not dc and g > gap_threshold
                cbs_ = not dc and not gs and (
                    _checkbox_between(hc[-1][-1], s) or
                    _unicode_cb_between(hc[-1][-1], s))
                if dc or gs or cbs_: hc.append([s])
                else: hc[-1].append(s)
                pc = cc
            bcr.append(hc)

        mc = max(len(row) for row in bcr)
        if mc == 1:
            asp = [s for b in vg for s in b]
            y0  = min(s["bbox"][1] for s in asp)
            y1  = max(s["bbox"][3] for s in asp)
            x0  = min(s["bbox"][0] for s in asp)
            ax  = rb if rb else max(s["bbox"][2] for s in asp)
            all_cl.append({"spans":asp,"sub_clusters":[asp],
                           "rect":pymupdf.Rect(x0,y0,max(ax,x0+4),y1),
                           "is_right":False})
        else:
            merged = [list(c) for c in bcr[0]]
            for row in bcr[1:]:
                for bc in row:
                    bx0 = bc[0]["bbox"][0]; bx1 = bc[-1]["bbox"][2]
                    bi, bo = None, 0
                    for i, mc2 in enumerate(merged):
                        mx0 = min(s["bbox"][0] for s in mc2)
                        mx1 = max(s["bbox"][2] for s in mc2)
                        ov = min(bx1,mx1)-max(bx0,mx0)
                        if ov > bo: bo = ov; bi = i
                    if bi is not None and bo > 4: merged[bi].extend(bc)
                    else: merged.append(list(bc))

            final = [{"spans":merged[0],"sub_clusters":[merged[0]]}]
            for cl in merged[1:]:
                ct = " ".join(s["text"] for s in cl).strip()
                pt = " ".join(s["text"] for s in final[-1]["spans"]).strip()
                px1 = max(s["bbox"][2] for s in final[-1]["spans"])
                cx0 = min(s["bbox"][0] for s in cl)
                g = cx0-px1
                if (len(ct)<4 or len(pt)<4) and g<60:
                    final[-1]["spans"].extend(cl)
                    final[-1]["sub_clusters"].append(cl)
                else:
                    final.append({"spans":list(cl),"sub_clusters":[cl]})

            n = len(final)
            for i, entry in enumerate(final):
                cl = entry["spans"]
                cx0 = min(s["bbox"][0] for s in cl)
                cx1 = max(s["bbox"][2] for s in cl)
                cy0 = min(s["bbox"][1] for s in cl)
                cy1 = max(s["bbox"][3] for s in cl)
                if i+1 < n:
                    ax = min(s["bbox"][0] for s in final[i+1]["spans"])-1.0
                elif rb: ax = rb
                else:    ax = cx1
                ir = (i == n-1 and cell_rect is not None
                      and cx0 > cell_rect.x0+cell_rect.width*0.6)
                all_cl.append({"spans":cl,"sub_clusters":entry["sub_clusters"],
                               "rect":pymupdf.Rect(cx0,cy0,max(ax,cx0+4),cy1),
                               "is_right":ir})
    return all_cl


# ── Post-translation helpers ──────────────────────────────────
_CITE_RE = re.compile(
    r"G\.L\.\s+c\.\s+[\d]+[A-Za-z]?(?:[,\s]+§\s*\d+[A-Za-z]*)?|§\s*\d+[A-Za-z]*",
    re.IGNORECASE)
_FUSED_RE = re.compile(
    r'\b(desde|hasta|para|por|con|del)(al|el|un|una|los|las|la|le)\b',
    re.IGNORECASE)

def _citation_fallback(orig, trans):
    missing = [m.group(0) for m in _CITE_RE.finditer(orig)
               if m.group(0) not in trans]
    if missing: trans = trans.rstrip()+" ("+", ".join(missing)+")"
    return trans

def _restore_caps_aware(orig, trans):
    letters = [c for c in orig if c.isalpha()]
    if letters and all(c.isupper() for c in letters): return trans.upper()
    return trans

def _fix_fused_words(text):
    text = _FUSED_RE.sub(lambda m: m.group(1)+' '+m.group(2), text)
    text = re.sub(r'\bmenor\s+([A-Ea-e])\.?\b',
                  lambda m: f"MENOR {m.group(1).upper()}", text)
    return text

def _should_never_translate(text):
    return bool(re.search(r'https?://|www\.', text.strip(), re.IGNORECASE))


# ════════════════════════════════════════════════════════════
# RC1 + FIX-3: PROTECTED DRAWING REDACTION
# ════════════════════════════════════════════════════════════

def _clip_redact_rect(rect, protected_drawings, margin=1.5):
    r = pymupdf.Rect(rect.x0+0.5, rect.y0+0.5, rect.x1-0.5, rect.y1-0.5)
    if r.width < 1 or r.height < 1: return None
    for pd in protected_drawings:
        if not r.intersects(pd): continue
        overlap = r & pd
        if overlap.is_empty: continue
        ow, oh = overlap.width, overlap.height
        if   oh < 3 and overlap.y0 <= r.y0+2: r.y0 = pd.y1+0.5
        elif oh < 3 and overlap.y1 >= r.y1-2: r.y1 = pd.y0-0.5
        elif ow < 3 and overlap.x0 <= r.x0+2: r.x0 = pd.x1+0.5
        elif ow < 3 and overlap.x1 >= r.x1-2: r.x1 = pd.x0-0.5
    if r.width < 1 or r.height < 1: return None
    return r


# ════════════════════════════════════════════════════════════
# CELL-CONTENT MAP BUILDER — FIX-9: in_table_cell flag
# ════════════════════════════════════════════════════════════

def _build_cell_map(blocks, table_cells, checkboxes, underlines, page_width):
    units = []
    for blk in blocks:
        if blk["type"] != 0: continue
        blk_rect = pymupdf.Rect(blk["bbox"])
        blk_cell = _find_tightest_table_cell(blk_rect, table_cells, 4.0)
        container  = blk_cell if blk_cell is not None else blk_rect
        in_table   = blk_cell is not None  # FIX-9: carry flag to every unit

        blk_lines = [ln for ln in blk["lines"]
                     if any(s["text"].strip() for s in ln["spans"])]
        if not blk_lines: continue

        paragraphs = _group_lines_into_paragraphs(blk_lines, container=container)

        for para_lines in paragraphs:
            para_spans = []
            for ln in para_lines:
                for span in ln["spans"]:
                    if span["text"].strip(): para_spans.append(span)
            if not para_spans: continue

            para_spans = _presplit_spans_at_checkboxes(para_spans)

            py0 = min(s["bbox"][1] for s in para_spans)
            py1 = max(s["bbox"][3] for s in para_spans)
            px0 = min(s["bbox"][0] for s in para_spans)
            px1 = max(s["bbox"][2] for s in para_spans)
            para_rect = pymupdf.Rect(px0, py0, px1, py1)

            col_clusters = _split_spans_into_columns(
                para_spans, container, blk_rect=para_rect,
                table_cells=table_cells, checkboxes=checkboxes)

            for cluster in col_clusters:
                cl_spans    = cluster["spans"]
                cl_rect     = cluster["rect"]
                is_right_cl = cluster["is_right"]

                fill_segments = _split_at_fill_fields(cl_spans)
                has_fills = any(seg["type"] == "fill" for seg in fill_segments)

                # ── helper to build a unit dict ───────────────────────
                def _make_unit(orig_text, sgs, ir, cl_r, spans, is_cen,
                               is_right, tp, preserve, cb, cg, icb, fl,
                               is_cont=False):
                    return {
                        "cell_rect":     ir,
                        "container_rect": container,
                        "orig_text":     orig_text,
                        "trans_text":    orig_text,
                        "style_groups":  sgs,
                        "is_centered":   is_cen,
                        "is_right":      is_right,
                        "top_pad":       tp,
                        "preserve":      preserve,
                        "checkbox":      cb,
                        "checkbox_gap":  cg,
                        "inline_cbs":    icb,
                        "all_spans":     spans,
                        "fill_lines":    fl,
                        "cl_rect":       cl_r,
                        "in_table_cell": in_table,      # FIX-9
                        "_is_continuation": is_cont,
                    }

                if not has_fills:
                    orig_text = " ".join(s["text"] for s in cl_spans)
                    orig_text = " ".join(orig_text.split())
                    if not orig_text.strip(): continue

                    sgs       = _spans_to_style_groups(cl_spans)
                    alignment = _detect_alignment(cl_spans, container)
                    is_cen    = alignment == 'center'
                    is_right  = is_right_cl or alignment == 'right'
                    al        = [c for c in orig_text if c.isalpha()]
                    is_hdr    = (al and all(c.isupper() for c in al)
                                 and is_cen and len(orig_text) < 100)
                    tp        = 3.0 if is_hdr else 0.0
                    preserve  = _should_never_translate(orig_text)

                    tu  = _union_rects([pymupdf.Rect(s["bbox"]) for s in cl_spans])
                    cb, cg = _find_associated_checkbox(tu, checkboxes)
                    icb = _find_inline_checkboxes(cl_spans, checkboxes)

                    ix0 = cb["rect"].x1+cg if cb else cl_rect.x0
                    # FIX-8c: extend to container right edge
                    ix1 = container.x1
                    ir  = pymupdf.Rect(ix0, cl_rect.y0,
                                       max(ix1, ix0+10), cl_rect.y1)
                    fl  = _get_fill_lines_for_cluster(cl_rect, container, underlines)
                    text_chunks = _split_long_text(orig_text, sgs)

                    if len(text_chunks) == 1:
                        units.append(_make_unit(
                            orig_text, sgs, ir, cl_rect, cl_spans,
                            is_cen, is_right, tp, preserve,
                            cb, cg, icb, fl))
                    else:
                        tc = sum(len(c) for c in text_chunks)
                        yc = ir.y0; ah = ir.height
                        for ci, chunk in enumerate(text_chunks):
                            ch_h = max(ah*len(chunk)/max(tc,1),
                                       _dominant_size(sgs)*1.5)
                            ch_r = pymupdf.Rect(ir.x0, yc, ir.x1, yc+ch_h)
                            units.append(_make_unit(
                                chunk, sgs, ch_r, cl_rect,
                                cl_spans if ci == 0 else [],
                                is_cen, is_right,
                                tp if ci == 0 else 0.0,
                                preserve,
                                cb  if ci == 0 else None,
                                cg  if ci == 0 else 0.0,
                                icb if ci == 0 else [],
                                fl  if ci == len(text_chunks)-1 else [],
                                is_cont=(ci > 0)))
                            yc += ch_h
                else:
                    for seg in fill_segments:
                        seg_spans = seg["spans"]
                        seg_rect  = _union_rects(
                            [pymupdf.Rect(s["bbox"]) for s in seg_spans])
                        if seg["type"] == "fill":
                            orig_text = " ".join(s["text"] for s in seg_spans)
                            units.append(_make_unit(
                                orig_text,
                                _spans_to_style_groups(seg_spans),
                                seg_rect, seg_rect, seg_spans,
                                False, False, 0.0, True,
                                None, 0.0, [], []))
                        else:
                            orig_text = " ".join(s["text"] for s in seg_spans)
                            orig_text = " ".join(orig_text.split())
                            if not orig_text.strip(): continue
                            sgs       = _spans_to_style_groups(seg_spans)
                            alignment = _detect_alignment(seg_spans, container)
                            is_cen    = alignment == 'center'
                            is_right  = is_right_cl or alignment == 'right'
                            preserve  = _should_never_translate(orig_text)
                            ix0 = seg_rect.x0
                            ix1 = min(seg_rect.x1+20, container.x1)
                            ir  = pymupdf.Rect(ix0, seg_rect.y0,
                                               max(ix1, ix0+10), seg_rect.y1)
                            fl  = _get_fill_lines_for_cluster(
                                seg_rect, container, underlines)
                            units.append(_make_unit(
                                orig_text, sgs, ir, seg_rect, seg_spans,
                                is_cen, is_right, 0.0, preserve,
                                None, 0.0, [], fl))

    units.sort(key=lambda u: (round(u["cell_rect"].y0, 1), u["cell_rect"].x0))
    return units


# ════════════════════════════════════════════════════════════
# COLUMN-AWARE Y-PUSH REFLOW  (FIX-9b)
# ════════════════════════════════════════════════════════════

def _build_column_index(units):
    """
    FIX-11b: Column band assignment with FROZEN widths and vertical
    proximity gating.

    Two problems with the previous implementation:
      (a) Bands were widened each time a new unit joined, so a wide
          paragraph early on would absorb all subsequent narrower units
          that happened to overlap at the edges, causing y_push to
          propagate across visually unrelated sections.
      (b) There was no vertical check — a unit 300pt below a prior
          unit was still assigned to the same column, inheriting push.

    Fixes:
      — Band x-range is FROZEN at creation time (first unit defines it).
      — A new unit only joins a band if it overlaps the ORIGINAL band
        x-range by ≥ 40% of its own width (tighter than before).
      — Vertical proximity gate: a unit only joins a band if the
        nearest prior unit in that band is within 4× its own height.
        This prevents paragraph sections separated by whitespace from
        sharing reflow state.

    Units with in_table_cell=True get column ID -1 (no reflow).
    """
    col_ids        = [-1] * len(units)
    # Each entry: (frozen_x0, frozen_x1, last_y1_in_col)
    col_meta       = []

    for i, u in enumerate(units):
        if u.get("in_table_cell", False):
            continue
        cr = u["cell_rect"]
        own_h = max(cr.height, 4.0)
        placed = False

        for cid, (fx0, fx1, last_y1) in enumerate(col_meta):
            # Vertical proximity: skip if too far below last unit
            if cr.y0 > last_y1 + own_h * 4:
                continue
            # Overlap with FROZEN band width
            overlap = min(cr.x1, fx1) - max(cr.x0, fx0)
            if cr.width > 0 and overlap >= cr.width * 0.40:
                col_ids[i] = cid
                # Update only the last_y1 — band x-range stays frozen
                col_meta[cid] = (fx0, fx1, cr.y1)
                placed = True
                break

        if not placed:
            col_ids[i] = len(col_meta)
            col_meta.append((cr.x0, cr.x1, cr.y1))

    return col_ids


# ════════════════════════════════════════════════════════════
# DIGITAL PAGE RECONSTRUCTION
# ════════════════════════════════════════════════════════════

def reconstruct_digital_page(out_page, use_verification=True):
    td = out_page.get_text("dict", flags=pymupdf.TEXTFLAGS_TEXT)
    blocks = [b for b in td["blocks"] if b["type"] == 0]
    if not blocks: return 0

    dm        = _classify_drawings(out_page)
    tc        = dm["table_cells"]
    ul        = dm["underlines"]
    cbs       = dm["checkboxes"]
    protected = dm["all_protected"]
    pw        = out_page.rect.width

    units = _build_cell_map(blocks, tc, cbs, ul, pw)
    if not units: return 0

    page_num    = out_page.number + 1
    unit_traces = []
    for i, u in enumerate(units):
        t = start_trace(page_num, i, u["orig_text"],
                        cell_rect=u["cell_rect"], is_preserved=u["preserve"])
        unit_traces.append(t)

    tidx  = [i for i, u in enumerate(units) if not u["preserve"]]
    trans = []
    for i in tidx:
        set_current_trace(unit_traces[i])
        trans.append(translate_one(units[i]["orig_text"]))
    set_current_trace(None)

    if use_verification and trans:
        to_t  = [units[i]["orig_text"] for i in tidx]
        trans = verify_page_translations(to_t, trans, batch_size=16)
        for j, t in enumerate(trans):
            trace_post_llama(tidx[j], t)

    for i, t in zip(tidx, trans):
        units[i]["trans_text"] = t

    for i, u in enumerate(units):
        if u["preserve"]: continue
        t = u["trans_text"]
        t = _citation_fallback(u["orig_text"], t)
        t = _restore_caps_aware(u["orig_text"], t)
        t = _fix_fused_words(t)
        u["trans_text"] = t
        trace_final_output(i, t)

    # ── Redaction phase ───────────────────────────────────────
    rul_keys = set()
    def _add_ul_redact(ui):
        r = ui["rect"]
        k = (round(r.x0,1),round(r.y0,1),round(r.x1,1),round(r.y1,1))
        if k not in rul_keys:
            clipped = _clip_redact_rect(
                pymupdf.Rect(r.x0, r.y0-0.5, r.x1, r.y1+1), protected)
            if clipped:
                out_page.add_redact_annot(clipped, fill=None)
            rul_keys.add(k)

    for u in units:
        for span in u["all_spans"]:
            sb  = span["bbox"]
            raw = pymupdf.Rect(sb[0], sb[1]-1, sb[2], sb[3]+1)
            clipped = _clip_redact_rect(raw, protected)
            if clipped:
                out_page.add_redact_annot(clipped, fill=None)
            for ui in _find_underlines_for_span(sb, ul):
                _add_ul_redact(ui)
        if not u["preserve"]:
            for fi in u.get("fill_lines", []):
                _add_ul_redact(fi)

    out_page.apply_redactions(images=pymupdf.PDF_REDACT_IMAGE_NONE)
    for w in list(out_page.widgets()):
        out_page.delete_widget(w)

    # ── Insertion phase with column-aware y-push reflow ───────
    # FIX-9b: build per-unit y-ceiling (next unit in same column)
    col_ids     = _build_column_index(units)
    col_y_push  = {}   # column_id -> accumulated y expansion

    # next-unit ceiling per unit for reflow mode
    next_y0 = [None] * len(units)
    for i in range(len(units) - 1):
        cid = col_ids[i]
        if cid < 0: continue
        for j in range(i+1, len(units)):
            if col_ids[j] == cid:
                next_y0[i] = units[j]["cell_rect"].y0
                break

    count = 0
    for idx, unit in enumerate(units):
        cid    = col_ids[idx]
        y_push = col_y_push.get(cid, 0.0) if cid >= 0 else 0.0
        m_y1   = next_y0[idx]

        tb, expansion = _insert_cell_unit(
            out_page, unit, max_y1=m_y1, y_push=y_push)

        count += 1

        # Accumulate expansion for subsequent units in same column
        if cid >= 0 and expansion > 0.5:
            col_y_push[cid] = col_y_push.get(cid, 0.0) + expansion

        if unit["preserve"]: continue
        clr = unit.get("cl_rect", unit["cell_rect"])
        for fi in unit.get("fill_lines", []):
            fr = fi["rect"]; fc_ = fi.get("color") or (0,0,0)
            fw = fi.get("width", 0.5)
            ny = (max(tb - fr.height*0.5, unit["cell_rect"].y0+2)
                  if fr.x0 >= clr.x1-5
                  else tb + max(fr.y0 - clr.y1, 1.0))
            try:
                out_page.draw_line(pymupdf.Point(fr.x0, ny),
                                   pymupdf.Point(fr.x1, ny),
                                   color=fc_, width=fw)
            except Exception: pass
    return count


# ── SCANNED pages ─────────────────────────────────────────────
def reconstruct_scanned_page_paddle(out_page, paddle_engine,
                                    use_verification=True):
    mat = pymupdf.Matrix(300/72, 300/72)
    pix = out_page.get_pixmap(matrix=mat)
    img_path = os.path.join(WORK_DIR, f"_ocr_tmp_{out_page.number}.png")
    pix.save(img_path)
    try: result = paddle_engine.ocr(img_path, cls=True)
    except Exception as e:
        print(f"   ⚠️  PaddleOCR failed: {e}"); return 0
    if not result or not result[0]: return 0
    sx = out_page.rect.width/pix.width; sy = out_page.rect.height/pix.height
    raw = []
    for line in result[0]:
        box, (text, conf) = line[0], line[1]
        if conf < OCR_CONFIDENCE or not text.strip(): continue
        if text.strip() == "X" or _is_blank_fill_line(text): continue
        xs = [p[0]*sx for p in box]; ys = [p[1]*sy for p in box]
        raw.append({"text": text,
                    "rect": pymupdf.Rect(min(xs),min(ys),max(xs),max(ys))})
    if not raw: return 0
    raw.sort(key=lambda u: (u["rect"].y0, u["rect"].x0))
    origs = [u["text"] for u in raw]
    trans = batch_translate(origs)
    if use_verification:
        trans = verify_page_translations(origs, trans, batch_size=16)
    final = [_fix_fused_words(_restore_caps_aware(o,_citation_fallback(o,t)))
             for o, t in zip(origs, trans)]
    count = 0
    for unit, t in zip(raw, final):
        r = unit["rect"]; sz = get_font_size({"size": r.height*0.72})
        out_page.add_redact_annot(
            pymupdf.Rect(r.x0, r.y0-1, r.x1, r.y1+1), fill=None)
        out_page.apply_redactions(images=pymupdf.PDF_REDACT_IMAGE_NONE)
        dsgs = [{"text":t,"family":"Helvetica, Arial, sans-serif",
                 "bold":False,"italic":False,"size":sz,"color":(0,0,0)}]
        _insert_cell_unit(out_page, {
            "cell_rect":r,"container_rect":r,"trans_text":t,
            "style_groups":dsgs,"is_centered":False,"is_right":False,
            "top_pad":0.0,"preserve":False,"fill_lines":[],
            "in_table_cell":False})
        count += 1
    return count

def reconstruct_scanned_page_tesseract(out_page, use_verification=True):
    try:
        import pytesseract; from PIL import Image as PILImage
    except ImportError:
        print("   ⚠️  Tesseract not available"); return 0
    mat = pymupdf.Matrix(300/72, 300/72)
    pix = out_page.get_pixmap(matrix=mat)
    img = PILImage.frombytes("RGB", (pix.width, pix.height), pix.samples)
    sx = out_page.rect.width/pix.width; sy = out_page.rect.height/pix.height
    data = pytesseract.image_to_data(img, lang="eng", config="--psm 6",
                                     output_type=pytesseract.Output.DICT)
    lm = {}
    for i, w in enumerate(data["text"]):
        if not w.strip() or int(data["conf"][i]) < 30: continue
        k = (data["block_num"][i], data["par_num"][i], data["line_num"][i])
        lm.setdefault(k, []).append(i)
    units = []
    for k in sorted(lm, key=lambda k: (min(data["top"][i]  for i in lm[k]),
                                        min(data["left"][i] for i in lm[k]))):
        idx = lm[k]; ws = " ".join(data["text"][i] for i in idx)
        if not ws.strip() or ws.strip() == "X" or _is_blank_fill_line(ws): continue
        r = pymupdf.Rect(
            min(data["left"][i])*sx,
            min(data["top"][i])*sy,
            max(data["left"][i]+data["width"][i])*sx,
            max(data["top"][i]+data["height"][i])*sy)
        units.append({"text": ws, "rect": r})
    if not units: return 0
    origs = [u["text"] for u in units]
    trans = batch_translate(origs)
    if use_verification:
        trans = verify_page_translations(origs, trans, batch_size=16)
    final = [_fix_fused_words(_restore_caps_aware(o,_citation_fallback(o,t)))
             for o, t in zip(origs, trans)]
    count = 0
    for unit, t in zip(units, final):
        r = unit["rect"]; sz = max(r.height*0.65-2.0, 5.0)
        out_page.add_redact_annot(
            pymupdf.Rect(r.x0, r.y0-1, r.x1, r.y1+1), fill=None)
        out_page.apply_redactions(images=pymupdf.PDF_REDACT_IMAGE_NONE)
        dsgs = [{"text":t,"family":"Helvetica, Arial, sans-serif",
                 "bold":False,"italic":False,"size":sz,"color":(0,0,0)}]
        _insert_cell_unit(out_page, {
            "cell_rect":r,"container_rect":r,"trans_text":t,
            "style_groups":dsgs,"is_centered":False,"is_right":False,
            "top_pad":0.0,"preserve":False,"fill_lines":[],
            "in_table_cell":False})
        count += 1
    return count


print("✅ Cell 9 v23 ready — FIX-1 through FIX-10 applied")
print("   FIX-9 : Hard cell boundary — table text clipped to drawn cell wall")
print("           in_table_cell flag on every unit; iterative font shrink")
print("   FIX-9b: Column-aware y-push reflow for free-text blocks")
print("   FIX-10: Checkbox threshold 20pt + square-shape guard (|w-h|<8)")

✅ Cell 9 v23 ready — FIX-1 through FIX-10 applied
   FIX-9 : Hard cell boundary — table text clipped to drawn cell wall
           in_table_cell flag on every unit; iterative font shrink
   FIX-9b: Column-aware y-push reflow for free-text blocks
   FIX-10: Checkbox threshold 20pt + square-shape guard (|w-h|<8)


# Translation diagnostics

In [ ]:
# ============================================================
# TRANSLATION DIAGNOSTICS — PIPELINE-INTEGRATED TRACING
#
# This cell provides two things:
#
#   1. TRACE CAPTURE: hooks into the actual pipeline (Cell 6/7/9)
#      to record what happened at each step during a real run.
#      No re-translation needed — data captured live.
#
#   2. REPORT GENERATION: after pipeline completes, generates
#      diagnostic tables and HTML reports from captured traces.
#
# Usage:
#   # Before running Cell 10 pipeline:
#   TRACE_ENABLED = True
#   traces.clear()
#
#   # After Cell 10 completes:
#   df = build_trace_report()
#   save_trace_html(df, "diagnostics.html")
#
# Each trace record captures:
#   - page number, unit index
#   - original English text (as chunked by Cell 9)
#   - NLLB raw output (before glossary)
#   - post-glossary output (after Cell 6 corrections)
#   - post-Llama output (after Cell 7 verification)
#   - final output (after post-processing in Cell 9)
#   - glossary terms found, which fired, which didn't
#   - whether text was preserved (URL, citation-only, etc.)
#   - cell rect coordinates (for spatial debugging)
# ============================================================

import pandas as pd
from collections import defaultdict
from IPython.display import display, HTML
import os

# ── Global trace storage ──────────────────────────────────────
TRACE_ENABLED = False
traces = []


class TranslationTrace:
    """
    Captures the state of one translation unit at each pipeline step.
    Created in Cell 9's _build_cell_map, populated as the unit
    flows through translate_one → glossary → Llama → post-process.
    """
    def __init__(self, page_num, unit_idx, original, cell_rect=None,
                 is_preserved=False):
        self.page_num      = page_num
        self.unit_idx      = unit_idx
        self.original      = original
        self.cell_rect     = cell_rect
        self.is_preserved  = is_preserved

        # Populated during pipeline
        self.nllb_raw      = None   # Step 4: raw NLLB output
        self.post_glossary = None   # Step 6: after glossary enforcement
        self.post_llama    = None   # Cell 7: after Llama verification
        self.final_output  = None   # Cell 9: after post-processing

        # Glossary analysis
        self.glossary_terms_found   = {}  # en_term → expected_es
        self.glossary_terms_correct = {}  # en_term → True/False (in NLLB raw)
        self.glossary_corrections   = {}  # en_term → what was changed

    def to_dict(self) -> dict:
        return {
            "page":             self.page_num,
            "unit":             self.unit_idx,
            "original":         self.original,
            "nllb_raw":         self.nllb_raw or "",
            "post_glossary":    self.post_glossary or "",
            "post_llama":       self.post_llama or "",
            "final_output":     self.final_output or "",
            "is_preserved":     self.is_preserved,
            "glossary_found":   len(self.glossary_terms_found),
            "glossary_correct": sum(self.glossary_terms_correct.values())
                                if self.glossary_terms_correct else 0,
            "glossary_fixed":   len(self.glossary_corrections),
            "nllb_changed":     self.nllb_raw != self.post_glossary
                                if self.nllb_raw and self.post_glossary else False,
            "llama_changed":    self.post_glossary != self.post_llama
                                if self.post_glossary and self.post_llama else False,
            "rect":             (f"({self.cell_rect.x0:.0f},{self.cell_rect.y0:.0f},"
                                 f"{self.cell_rect.x1:.0f},{self.cell_rect.y1:.0f})")
                                if self.cell_rect else "",
        }


def start_trace(page_num, unit_idx, original, cell_rect=None,
                is_preserved=False):
    """Create a new trace record. Called from Cell 9."""
    if not TRACE_ENABLED:
        return None
    t = TranslationTrace(page_num, unit_idx, original,
                         cell_rect, is_preserved)
    traces.append(t)
    return t


# ── Hooks for Cell 6 ─────────────────────────────────────────
# These are called from within translate_one to capture
# intermediate states. They're no-ops when tracing is off.

_current_trace = None  # set by Cell 9 before calling translate_one

def set_current_trace(trace):
    """Cell 9 calls this before translate_one for each unit."""
    global _current_trace
    _current_trace = trace

def trace_nllb_raw(text):
    """Called after Step 4 (NLLB raw output) in translate_one."""
    if _current_trace and TRACE_ENABLED:
        _current_trace.nllb_raw = text

def trace_post_glossary(text):
    """Called after Step 6 (glossary enforcement) in translate_one."""
    if _current_trace and TRACE_ENABLED:
        _current_trace.post_glossary = text

def trace_glossary_analysis(original, nllb_output):
    """
    Called during glossary enforcement to record which terms
    were found, which NLLB got right, and which needed fixing.
    """
    if not _current_trace or not TRACE_ENABLED:
        return
    try:
        glossary = GLOSSARY_ES
    except NameError:
        return
    try:
        skip_terms = {k.lower() for k in CONTEXT_DEPENDENT_ES.keys()}
    except NameError:
        skip_terms = set()

    orig_lower = original.lower()
    for en_term in sorted(glossary.keys(), key=len, reverse=True):
        if en_term not in orig_lower or en_term in skip_terms:
            continue
        expected = glossary[en_term]
        alts = [a.strip() for a in expected.split(",") if a.strip()]
        if not alts:
            continue

        _current_trace.glossary_terms_found[en_term] = expected

        # Check if any correct alternative is in NLLB output
        found = False
        for alt in alts:
            stem = alt[:6].lower() if len(alt) >= 6 else alt.lower()
            if stem in nllb_output.lower():
                found = True; break

        _current_trace.glossary_terms_correct[en_term] = found


# ── Hooks for Cell 7 (Llama) ─────────────────────────────────

def trace_post_llama(unit_idx, text):
    """
    Called after Llama verification to record the result.
    Maps by unit_idx since Llama processes in batches.
    """
    if not TRACE_ENABLED:
        return
    # Find the trace by unit_idx for the current page
    for t in reversed(traces):
        if t.unit_idx == unit_idx and t.post_llama is None:
            t.post_llama = text
            return


def trace_final_output(unit_idx, text):
    """Called after Cell 9 post-processing (caps, fused words, etc.)."""
    if not TRACE_ENABLED:
        return
    for t in reversed(traces):
        if t.unit_idx == unit_idx and t.final_output is None:
            t.final_output = text
            return


# ════════════════════════════════════════════════════════════
# REPORT GENERATION
# ════════════════════════════════════════════════════════════

def build_trace_report() -> pd.DataFrame:
    """
    Build a DataFrame from all captured traces.
    Call this after the Cell 10 pipeline completes.
    """
    if not traces:
        print("⚠️  No traces captured. Set TRACE_ENABLED = True before running pipeline.")
        return pd.DataFrame()

    rows = [t.to_dict() for t in traces]
    df = pd.DataFrame(rows)

    # Summary stats
    total     = len(df)
    preserved = df["is_preserved"].sum()
    translated = total - preserved
    nllb_changed = df["nllb_changed"].sum()
    llama_changed = df["llama_changed"].sum()
    glossary_fixes = df["glossary_fixed"].sum()

    print("=" * 70)
    print("  TRANSLATION PIPELINE DIAGNOSTIC SUMMARY")
    print("=" * 70)
    print(f"  Total units processed:     {total}")
    print(f"  Preserved (URLs, etc.):    {preserved}")
    print(f"  Translated:                {translated}")
    print(f"  Glossary corrections:      {glossary_fixes}")
    print(f"  Units changed by glossary: {nllb_changed}")
    print(f"  Units changed by Llama:    {llama_changed}")

    if translated > 0:
        terms_found   = df["glossary_found"].sum()
        terms_correct = df["glossary_correct"].sum()
        print(f"\n  Glossary terms found in source:  {terms_found}")
        print(f"  NLLB got correct (pre-glossary): {terms_correct}")
        if terms_found > 0:
            print(f"  NLLB term accuracy:              "
                  f"{terms_correct/terms_found*100:.1f}%")

    print("=" * 70)
    return df


def show_changes_only(df: pd.DataFrame = None) -> pd.DataFrame:
    """Show only units where something changed during the pipeline."""
    if df is None:
        df = build_trace_report()
    if df.empty:
        return df

    changed = df[df["nllb_changed"] | df["llama_changed"]].copy()
    if changed.empty:
        print("✅ No changes detected — NLLB output was used as-is everywhere.")
        return changed

    print(f"\n📋 {len(changed)} units were modified during pipeline:\n")
    for _, row in changed.iterrows():
        print(f"  Page {row['page']}, Unit {row['unit']}:")
        print(f"    EN:      {row['original'][:100]}")
        print(f"    NLLB:    {row['nllb_raw'][:100]}")
        if row["nllb_changed"]:
            print(f"    +Gloss:  {row['post_glossary'][:100]}")
        if row["llama_changed"]:
            print(f"    +Llama:  {row['post_llama'][:100]}")
        print(f"    Final:   {row['final_output'][:100]}")
        print()

    return changed


def show_glossary_analysis(df: pd.DataFrame = None) -> pd.DataFrame:
    """
    Show per-term glossary analysis across all traced units.
    Which terms did NLLB get right? Which needed correction?
    """
    if not traces:
        print("⚠️  No traces. Run pipeline with TRACE_ENABLED = True first.")
        return pd.DataFrame()

    rows = []
    for t in traces:
        if t.is_preserved:
            continue
        for en_term, expected_es in t.glossary_terms_found.items():
            nllb_correct = t.glossary_terms_correct.get(en_term, False)
            was_fixed    = en_term in t.glossary_corrections
            rows.append({
                "page":          t.page_num,
                "english_term":  en_term,
                "expected_es":   expected_es[:50],
                "nllb_correct":  "✅" if nllb_correct else "❌",
                "glossary_fixed": "✅" if was_fixed else "—",
                "context":       t.original[:60],
            })

    df = pd.DataFrame(rows) if rows else pd.DataFrame()

    if not df.empty:
        # Aggregate by term
        agg = df.groupby("english_term").agg(
            occurrences=("english_term", "count"),
            nllb_correct=("nllb_correct", lambda x: (x == "✅").sum()),
            glossary_fixed=("glossary_fixed", lambda x: (x == "✅").sum()),
        ).reset_index()
        agg["nllb_accuracy"] = (agg["nllb_correct"] / agg["occurrences"]
                                 * 100).round(1).astype(str) + "%"
        agg = agg.sort_values("nllb_correct", ascending=True)

        print("\n📊 GLOSSARY TERM ACCURACY (per term across all pages):\n")
        print(agg.to_string(index=False))
        print()

    return df


def save_trace_html(df: pd.DataFrame = None, path: str = None):
    """
    Save a color-coded HTML report of all trace data.
    Green = unchanged, Yellow = glossary corrected, Blue = Llama corrected.
    """
    if df is None:
        df = build_trace_report()
    if df.empty:
        print("⚠️  No data to export.")
        return

    if path is None:
        path = os.path.join(WORK_DIR, "translation_diagnostics.html")

    display_cols = ["page", "unit", "original", "nllb_raw",
                    "post_glossary", "post_llama", "final_output",
                    "glossary_found", "nllb_changed", "llama_changed"]
    export_df = df[display_cols].copy()

    def _row_color(row):
        if row.get("llama_changed"):
            return ["background-color: #cce5ff"] * len(row)  # blue
        elif row.get("nllb_changed"):
            return ["background-color: #fff3cd"] * len(row)  # yellow
        else:
            return ["background-color: #d4edda"] * len(row)  # green

    styled = (
        export_df.style
        .apply(_row_color, axis=1)
        .set_table_styles([
            {"selector": "th", "props": [
                ("background-color", "#343a40"), ("color", "white"),
                ("padding", "8px 12px"), ("font-size", "13px"),
            ]},
            {"selector": "td", "props": [
                ("padding", "6px 10px"), ("font-size", "11px"),
                ("border-bottom", "1px solid #dee2e6"),
                ("max-width", "300px"), ("overflow", "hidden"),
                ("text-overflow", "ellipsis"), ("white-space", "nowrap"),
            ]},
        ])
        .set_caption("Translation Pipeline Diagnostics")
    )

    html = styled.to_html()
    with open(path, "w", encoding="utf-8") as f:
        f.write(html)
    print(f"📄 Diagnostics saved → {path}")


def compare_pages(page_num: int = None):
    """
    Show all units for a specific page, or summary per page.
    Useful for debugging layout issues on specific pages.
    """
    if not traces:
        print("⚠️  No traces captured.")
        return

    if page_num is not None:
        page_traces = [t for t in traces if t.page_num == page_num]
        if not page_traces:
            print(f"⚠️  No traces for page {page_num}")
            return

        print(f"\n📄 Page {page_num}: {len(page_traces)} units\n")
        for t in page_traces:
            marker = "🔒" if t.is_preserved else "📝"
            changed = ""
            if t.nllb_raw != t.post_glossary:
                changed += " [glossary]"
            if t.post_glossary != t.post_llama:
                changed += " [llama]"
            if not changed:
                changed = " [unchanged]"

            print(f"  {marker} Unit {t.unit_idx}: {t.original[:80]}")
            if not t.is_preserved:
                print(f"       → {t.final_output[:80]}{changed}")
            print()
    else:
        # Summary per page
        page_stats = defaultdict(lambda: {"total":0, "preserved":0,
                                           "glossary":0, "llama":0})
        for t in traces:
            ps = page_stats[t.page_num]
            ps["total"] += 1
            if t.is_preserved:
                ps["preserved"] += 1
            if t.nllb_raw and t.post_glossary and t.nllb_raw != t.post_glossary:
                ps["glossary"] += 1
            if t.post_glossary and t.post_llama and t.post_glossary != t.post_llama:
                ps["llama"] += 1

        print("\n📊 PER-PAGE SUMMARY:\n")
        print(f"  {'Page':<6} {'Units':<8} {'Preserved':<12} "
              f"{'Glossary Fix':<14} {'Llama Fix':<10}")
        print("  " + "-" * 50)
        for pg in sorted(page_stats):
            ps = page_stats[pg]
            print(f"  {pg:<6} {ps['total']:<8} {ps['preserved']:<12} "
                  f"{ps['glossary']:<14} {ps['llama']:<10}")


print("✅ Diagnostics ready")
print("   To enable tracing, set TRACE_ENABLED = True before running pipeline")
print()
print("   After pipeline completes:")
print("     df = build_trace_report()          # summary + DataFrame")
print("     show_changes_only(df)              # only modified units")
print("     show_glossary_analysis()           # per-term NLLB accuracy")
print("     compare_pages(1)                   # debug specific page")
print("     save_trace_html(df)                # export HTML report")

✅ Diagnostics ready
   To enable tracing, set TRACE_ENABLED = True before running pipeline

   After pipeline completes:
     df = build_trace_report()          # summary + DataFrame
     show_changes_only(df)              # only modified units
     show_glossary_analysis()           # per-term NLLB accuracy
     compare_pages(1)                   # debug specific page
     save_trace_html(df)                # export HTML report


In [ ]:
TRACE_ENABLED = True
traces.clear()

# Cell 10 - main pipeline

In [ ]:
# ============================================================
# CELL 10: MAIN PIPELINE LOOP
# ============================================================
import os
from pathlib import Path
from google.colab import files

# PaddleOCR is optional — fall back to Tesseract if unavailable
try:
    from paddleocr import PaddleOCR
except ImportError:
    PaddleOCR = None

src_doc = pymupdf.open(INPUT_PDF)
out_doc = pymupdf.open()
out_doc.insert_pdf(src_doc)
src_doc.close()

paddle_engine  = None
digital_pages  = 0
scanned_pages  = 0
blank_pages    = 0
skipped_pages  = 0
failed_pages   = 0

for pg in range(len(out_doc)):
    out_page = out_doc[pg]
    try:
        info = classify_page(out_page)
    except Exception as e:
        print(f"\n⚠️  Page {pg+1}/{len(out_doc)}: classify_page failed — {e}")
        failed_pages += 1
        continue

    try:
        if info["page_type"] == "BLANK":
            print(f"\n⬜ Page {pg+1}/{len(out_doc)}: BLANK — skipping")
            blank_pages += 1

        elif info["page_type"] == "DIGITAL":
            print(f"\n📄 Page {pg+1}/{len(out_doc)}: DIGITAL "
                  f"({info['span_count']} spans, {info['drawings']} drawings, "
                  f"{info['img_coverage']}% image)")
            count = reconstruct_digital_page(out_page)
            print(f"   ✅ Translated & reinserted {count} units")
            digital_pages += 1

        elif info["is_scanned"]:
            if is_content_image(info):
                print(f"\n🖼️  Page {pg+1}/{len(out_doc)}: IMAGE-ONLY — preserving as-is")
                skipped_pages += 1
                continue

            print(f"\n📷 Page {pg+1}/{len(out_doc)}: SCANNED "
                  f"({info['span_count']} spans, {info['images']} images, "
                  f"{info['img_coverage']}% image)")

            if PaddleOCR is not None:
                if paddle_engine is None:
                    print("   ⏳ Initializing PaddleOCR...")
                    paddle_engine = PaddleOCR(
                        use_textline_orientation=True, lang="en"
                    )
                count = reconstruct_scanned_page_paddle(out_page, paddle_engine)
                print(f"   ✅ PaddleOCR: translated {count} regions")
            else:
                print("   ⚠️  PaddleOCR not available, falling back to Tesseract")
                count = reconstruct_scanned_page_tesseract(out_page)
                print(f"   ✅ Tesseract: translated {count} regions")

            scanned_pages += 1

    except Exception as e:
        print(f"   ❌ Page {pg+1}/{len(out_doc)}: reconstruction failed — {e}")
        failed_pages += 1
        continue

# Save with garbage collection and deflate compression.
# garbage=4: removes all unreferenced objects left by redactions.
# deflate=True: recompresses content streams for smaller file size.
input_stem = Path(INPUT_PDF).stem
OUTPUT_PDF = os.path.join(WORK_DIR, f"{input_stem}_translated.pdf")
out_doc.save(OUTPUT_PDF, garbage=4, deflate=True)
out_doc.close()

print(f"\n{'='*60}")
print("🎉 PIPELINE COMPLETE")
print(f"   Digital pages:      {digital_pages}")
print(f"   Scanned pages:      {scanned_pages}")
print(f"   Blank pages:        {blank_pages}")
print(f"   Image-only skipped: {skipped_pages}")
if failed_pages:
    print(f"   ⚠️  Failed pages:    {failed_pages}")
print(f"   Output: {OUTPUT_PDF}")
print(f"{'='*60}")

files.download(OUTPUT_PDF)
print(f"✅ Downloaded: {Path(OUTPUT_PDF).name}")


📄 Page 1/1: DIGITAL (60 spans, 28 drawings, 0.1% image)
   📋 Document mode: verifying all 27 spans
   ✅ Batch 1/2: verified 16 spans (14 glossary terms injected)
   ✅ Batch 2/2: verified 11 spans (16 glossary terms injected)
   ✅ Translated & reinserted 29 units

🎉 PIPELINE COMPLETE
   Digital pages:      1
   Scanned pages:      0
   Blank pages:        0
   Image-only skipped: 0
   Output: /content/courtaccess_output/test7 (1)_translated.pdf


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

✅ Downloaded: test7 (1)_translated.pdf


In [ ]:
# Full summary + DataFrame
df = build_trace_report()
save_trace_html(df)

  TRANSLATION PIPELINE DIAGNOSTIC SUMMARY
  Total units processed:     278
  Preserved (URLs, etc.):    9
  Translated:                269
  Glossary corrections:      0
  Units changed by glossary: 3
  Units changed by Llama:    161

  Glossary terms found in source:  197
  NLLB got correct (pre-glossary): 111
  NLLB term accuracy:              56.3%
📄 Diagnostics saved → /content/courtaccess_output/translation_diagnostics.html


In [ ]:
# Per-term analysis: which glossary terms does NLLB get right?
show_glossary_analysis()

# Cell 11 - visual

In [ ]:
# ============================================================
# CELL 11: VISUAL COMPARISON & DOWNLOAD
# Renders page 1 of original vs translated side-by-side for a
# quick visual sanity check, then downloads the output PDF.
# ============================================================

def show_comparison(original_path, translated_path, page_num=0, dpi=120):
    zoom = dpi / 72
    mat  = pymupdf.Matrix(zoom, zoom)
    orig_doc  = pymupdf.open(original_path)
    trans_doc = pymupdf.open(translated_path)
    pg        = min(page_num, len(orig_doc)-1, len(trans_doc)-1)

    orig_pix  = orig_doc[pg].get_pixmap(matrix=mat)
    trans_pix = trans_doc[pg].get_pixmap(matrix=mat)
    orig_img  = Image.frombytes("RGB", (orig_pix.width,  orig_pix.height),  orig_pix.samples)
    trans_img = Image.frombytes("RGB", (trans_pix.width, trans_pix.height), trans_pix.samples)

    max_h  = max(orig_img.height, trans_img.height)
    canvas = Image.new("RGB", (orig_img.width + trans_img.width + 20, max_h), (200, 200, 200))
    canvas.paste(orig_img,  (0, 0))
    canvas.paste(trans_img, (orig_img.width + 20, 0))

    draw = ImageDraw.Draw(canvas)
    draw.text((10, 5),                      "ORIGINAL (English)",   fill=(255, 0, 0))
    draw.text((orig_img.width + 30, 5),     "TRANSLATED (Spanish)", fill=(0, 128, 0))

    out_path = os.path.join(WORK_DIR, "comparison.png")
    canvas.save(out_path)
    display(IPImage(out_path))
    orig_doc.close()
    trans_doc.close()

print("🖼️  Page 1 — Original vs Translated:")
show_comparison(INPUT_PDF, OUTPUT_PDF, page_num=2)

# files.download(OUTPUT_PDF)
# print("✅ Downloaded: courtaccess_translated.pdf")